
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Deploying an Observable Agent
#### Duration: 30 minutes

## Overview

This demo progressively builds a production-ready agent with deployment, observability, and tool integration in a single flow. You will start with a bare-bones agent deployed as a Databricks App via Declarative Automation Bundles (DABs), layer on MLflow tracing for full request-level visibility, and then connect it to a SQL tool through the Model Context Protocol (MCP). By the end, every agent interaction (including tool calls) will produce structured MLflow traces you can inspect in the Databricks UI.

## Learning Objectives

By the end of this demonstration, you will be able to:

1. Deploy an agent on Databricks Apps using DABs
2. Add MLflow tracing for observability
3. Integrate tools through a Databricks-provided MCP
4. Verify traces in the MLflow UI

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Interactive Diagrams</strong>
<p style="margin: 8px 0 0 0; color: #333;">The architecture diagrams in this notebook are interactive. Click on any component to highlight it and view details about its role, code, and configuration in the panel below the diagram.</p>
</div>
</div>
</div>

## A. Required Compute Environment

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Serverless Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

<div style="
border-left: 4px solid #F44336;
background: #FFEBEE;
padding: 14px 18px;
border-radius: 4px;
margin: 16px 0;
">
<strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">GIT FOLDER REQUIREMENT</strong>
<div style="color:#333;">

The Declarative Automation Bundles (DABs) UI is only available when you are working inside a Git folder in the workspace.

Before opening the DABs UI, make sure you have:
- Created or cloned a Git folder in the workspace with all associated labs files
- Git is outside the scope of this lab. For more information visit the Databricks Git folders (<a href="https://docs.databricks.com/aws/en/repos/" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/repos/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/repos/" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>) documentation.
- **If you are working in Vocareum:** Vocareum does not automatically come with a git folder as a part of the workspace creation. Please use the official documentation to set up Git Folders if working in **Databricks Workspace**.
</div>
</div>

## B. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-02

## C. Define and Deploy a Basic Agent

In this section you will create the files needed to deploy a simple chatbot as a Databricks App. The agent uses the OpenAI Agents SDK with its client redirected to a Unity Gateway model service.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Copy and paste the code below to replace the contents of the corresponding files inside the <strong><code>simple-agent</code></strong> folder.</p>
</div>
</div>
</div>

### C1. Project Files

The basic agent requires six files. Select a component in the diagram to highlight its code, or select a tab to view each file and its role.

<style>
#c1-arch { width:100%; margin:12px 0 0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; color:#0b2026; }
#c1-arch .dia { margin:0 8px; background:#fff; border:1px solid #DCE0E2; border-radius:14px; padding:12px; box-shadow:0 4px 18px rgba(27,49,57,.06); }
#c1-arch svg { display:block; width:100%; height:auto; }
#c1-arch svg text { font-family:inherit; fill:#0b2026; }
#c1-arch .cont rect.box { fill:#F9F7F4; stroke:#DCE0E2; stroke-width:1.5; }
#c1-arch .cont .hdr { cursor:pointer; }
#c1-arch .cont .hdr text { font-size:13px; font-weight:700; letter-spacing:1.4px; fill:#618794; }
#c1-arch .cont .hdr:hover text, #c1-arch .cont.sel .hdr text { fill:#FF5F46; }
#c1-arch .nd { cursor:pointer; }
#c1-arch .nd rect.box { fill:#fff; stroke:#C9D0D3; stroke-width:1.5; transition:fill .15s, stroke .15s; }
#c1-arch .nd:hover rect.box { stroke:#1B5162; }
#c1-arch .nd .t { font-size:16px; font-weight:700; }
#c1-arch .nd .s { font-size:13px; fill:#618794; }
#c1-arch .nd.mono .t { font-family:ui-monospace,Menlo,Consolas,monospace; font-size:15px; }
#c1-arch .nd.small .t { font-size:13.5px; }
#c1-arch .nd.new rect.box { stroke:#FF5F46; stroke-width:2.2; }
#c1-arch .nd.sel rect.box { fill:#1B5162; stroke:#1B5162; }
#c1-arch .nd.sel .t, #c1-arch .nd.sel .s { fill:#fff; }
#c1-arch .pill rect { fill:#fff; stroke:#1B5162; stroke-width:1.2; }
#c1-arch .pill text { font-size:10.5px; font-weight:700; letter-spacing:.6px; fill:#1B5162; font-family:ui-monospace,Menlo,Consolas,monospace; }
#c1-arch .ed path { fill:none; stroke:#9AABB1; stroke-width:2; }
#c1-arch .ed.new path { stroke:#FF5F46; }
#c1-arch .ed text { font-size:12px; fill:#618794; font-family:ui-monospace,Menlo,Consolas,monospace; }
#c1-arch .ed.new text { fill:#FF5F46; }
#c1-arch .panel { margin-top:14px; border:1px solid #DCE0E2; border-radius:12px; overflow:hidden; background:#fff; }
#c1-arch .tabs { display:flex; flex-wrap:wrap; align-items:flex-end; gap:2px; background:#F9F7F4; border-bottom:1px solid #DCE0E2; padding:6px 8px 0; }
#c1-arch .tab { padding:8px 13px; font-size:13px; font-weight:600; color:#618794; border-radius:7px 7px 0 0; cursor:pointer; display:flex; gap:6px; align-items:center; font-family:ui-monospace,Menlo,Consolas,monospace; }
#c1-arch .tab.same { color:#AEB8BC; }
#c1-arch .tab.on { background:#fff; color:#0b2026; box-shadow:0 -1px 0 #DCE0E2, 1px 0 0 #DCE0E2, -1px 0 0 #DCE0E2; }
#c1-arch .tab .d { width:7px; height:7px; border-radius:50%; background:#FF5F46; }
#c1-arch .tab .n { font-size:10.5px; font-weight:700; color:#fff; background:#1B5162; border-radius:999px; padding:1px 6px; }
#c1-arch .grow { flex:1; }
#c1-arch .cp { display:inline-flex; flex:none; white-space:nowrap; min-width:84px; justify-content:center; align-items:center; gap:6px; margin-bottom:6px; border:1px solid #1B3139; background:#1B3139; color:#fff; padding:6px 12px; border-radius:6px; font-size:13px; font-weight:600; cursor:pointer; }
#c1-arch .cp.ok { background:#00A972; border-color:#00A972; }
#c1-arch .split { display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; align-items:start; }
#c1-arch .split.full { grid-template-columns:1fr; }
#c1-arch .split.full .side { display:none; }
#c1-arch .side { border-right:1px solid #EEEDE9; background:#fff; min-width:0; overflow-wrap:anywhere; }
#c1-arch .side code { overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
@media (min-width: 821px) { #c1-arch .split:not(.full) { align-items:stretch; grid-template-rows:minmax(420px, auto); } #c1-arch .split:not(.full) .code { height:0; min-height:100%; max-height:none; } }
@media (max-width: 820px) { #c1-arch .split { grid-template-columns:1fr; } #c1-arch .side { border-right:none; border-bottom:1px solid #EEEDE9; max-height:none; } }
#c1-arch .desc { padding:16px 20px 6px; }
#c1-arch .desc p { font-size:13pt; line-height:1.6; color:#333; margin:0 0 12px 0; }
#c1-arch .desc ul { margin:6px 0 10px 18px; padding-left:0; font-size:13pt; line-height:1.6; color:#333; }
#c1-arch .desc li { margin-bottom:4px; }
#c1-arch .chg { border-left:4px solid #1B5162; background:rgba(27,81,98,.06); border-radius:0 6px 6px 0; padding:10px 14px; margin:0 0 12px 0; font-size:12.5pt; line-height:1.55; }
#c1-arch .chg ul { margin:6px 0 0 20px; }
#c1-arch .code { font-size:13px; line-height:1.65; padding:10px 0; max-height:480px; overflow-y:auto; overflow-x:hidden; background:#fcfcfb; font-family:ui-monospace,Menlo,Consolas,monospace; color:#383a42; }
#c1-arch .ln { display:flex; white-space:pre-wrap; overflow-wrap:anywhere; border-left:3px solid transparent; transition:opacity .2s, background .2s; }
#c1-arch .ln > span:last-child { flex:1; min-width:0; padding-right:16px; }
#c1-arch .ln .g { width:40px; flex:none; text-align:right; padding-right:14px; color:#C4CBCE; user-select:none; }
#c1-arch .ln.add { background:rgba(0,169,114,.13); border-left-color:#00A972; }
#c1-arch .ln.add .g { color:#00A972; font-weight:700; }
#c1-arch .code.focus .ln { opacity:.32; }
#c1-arch .code.focus .ln.hit { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
#c1-arch .code.focus .ln.hit.add { background:rgba(0,169,114,.16); border-left-color:#00A972; }
#c1-arch .t-k { color:#a626a4 } #c1-arch .t-f { color:#4078f2 } #c1-arch .t-s { color:#50a14f } #c1-arch .t-n { color:#986801 } #c1-arch .t-c { color:#a0a1a7; font-style:italic } #c1-arch .t-d { color:#c18401 } #c1-arch .t-y { color:#e45649 }
</style>
<div id="c1-arch">
  <div class="dia"><svg id="c1-svg"></svg></div>
  <div class="panel"><div class="tabs" id="c1-tabs"></div><div class="split" id="c1-split"><div class="side" id="c1-desc"></div><div class="code" id="c1-code"></div></div></div>
</div>
<script>
(function () {
  var RAW = {"C1": [{"name": "agent.py", "desc": "<p>This file defines the agent and an async function to run it. The key integration point is <code>set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))<\/code>, which routes all model calls through Databricks instead of the public OpenAI service.<\/p><p><strong>Key components:<\/strong><\/p><ul><li><code>Agent<\/code> and <code>Runner<\/code>: Define and execute the agent workflow from the OpenAI Agents SDK<\/li><li><code>AsyncDatabricksOpenAI<\/code>: An async client that calls Unity Gateway model services using the OpenAI-compatible interface<\/li><li><code>set_default_openai_api(\"responses\")<\/code>: Sends requests through the Responses API, which supports tool calls with reasoning models<\/li><li><code>MODEL_SERVICE<\/code>: The model service name, read from an environment variable injected by the DAB at deploy time<\/li><\/ul>", "changes": "", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport os\n\n# Read the model service name injected by Databricks Apps\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\n\n# Point the OpenAI Agents SDK at Databricks instead of OpenAI\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Define the agent\nagent = Agent(\n    name=\"Simple Chatbot\",\n    instructions=\"You are a helpful assistant. Answer the user's questions clearly and concisely.\",\n    model=MODEL_SERVICE,\n)\n\n\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent with a list of messages and return the final reply.\"\"\"\n    result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "server.py", "desc": "<p>This FastAPI server exposes two endpoints: <code>GET /<\/code> returns a chat HTML page, and <code>POST /invocations<\/code> forwards conversation messages to the agent and returns the reply. It listens on the port Databricks Apps sets in <code>DATABRICKS_APP_PORT<\/code>.<\/p><p>This demo uses a small hand-written server so each piece is visible. For production agents, the <a href=\"https://github.com/databricks/app-templates/tree/main/agent-openai-agents-sdk\" target=\"_blank\" style=\"color: #1976d2; text-decoration: underline;\">agent app template<\/a> uses MLflow <code>AgentServer<\/code>, which adds the standard Responses API, streaming, and a built-in chat UI.<\/p>", "changes": "", "code": "import os\nimport uvicorn\nfrom pathlib import Path\n\nfrom fastapi import FastAPI\nfrom fastapi.responses import HTMLResponse\nfrom pydantic import BaseModel\n\nfrom agent import run_agent\n\napp = FastAPI()\n\nCHAT_UI = Path(__file__).parent / \"chat.html\"\n\n\nclass ChatRequest(BaseModel):\n    input: list[dict]\n\n\n@app.get(\"/\", response_class=HTMLResponse)\nasync def root():\n    return CHAT_UI.read_text()\n\n\n@app.post(\"/invocations\")\nasync def invocations(request: ChatRequest):\n    reply = await run_agent(request.input)\n    return {\"output\": reply}\n\n\ndef main():\n    uvicorn.run(\"server:app\", host=\"0.0.0.0\", port=int(os.getenv(\"DATABRICKS_APP_PORT\", 8000)))\n\n\nif __name__ == \"__main__\":\n    main()"}, {"name": "pyproject.toml", "desc": "<p>Standard Python project metadata and dependency declaration. The <code>[project.scripts]<\/code> entry creates the <code>start-server<\/code> CLI command referenced by <code>app.yaml<\/code>.<\/p>", "changes": "", "code": "[project]\nname = \"simple-chatbot\"\nversion = \"0.1.0\"\nrequires-python = \">=3.11\"\ndependencies = [\n    \"openai-agents>=0.6.0\",\n    \"mcp<2\",\n    \"databricks-openai==0.17.1\",\n    \"databricks-sdk==0.145.0\",\n    \"fastapi>=0.115.0\",\n    \"uvicorn>=0.34.0\",\n    \"uv\",\n]\n\n[build-system]\nrequires = [\"hatchling\"]\nbuild-backend = \"hatchling.build\"\n\n[tool.hatch.build.targets.wheel]\ninclude = [\"agent.py\", \"server.py\"]\n\n[project.scripts]\nstart-server = \"server:main\""}, {"name": "app.yaml", "desc": "<p>The Databricks App manifest. It declares which environment variables to inject (resolved from DAB resources) and the startup command.<\/p>", "changes": "", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}, {"name": "databricks.yml", "desc": "<p>The Declarative Automation Bundle (DAB) manifest. It names the app, grants it <code>EXECUTE<\/code> on the course model service <code>workspace.agent_deployment.agent-model<\/code>, and targets the <code>dev<\/code> deployment mode. The app name uses bundle substitutions (<code>${workspace.current_user.id}<\/code>, <code>${bundle.target}<\/code>) so each user gets a unique app.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file is only needed for DABs-based deployment. If you are deploying via the Apps UI instead, skip this file and configure the app resources manually on the app Overview page.<\/div>", "changes": "", "code": "bundle:\n  name: simple_chatbot\n\nresources:\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n\ntargets:\n  dev:\n    mode: development\n    default: true"}, {"name": "requirements.txt", "desc": "<p>Databricks Apps automatically runs <code>pip install -r requirements.txt<\/code> during deployment. Installing <code>uv<\/code> here ensures the <code>uv run start-server<\/code> command in <code>app.yaml<\/code> works correctly.<\/p>", "changes": "", "code": "uv"}], "D1": [{"name": "agent.py", "desc": "<p>Replace the contents of <strong>agent.py<\/strong> with the version below. The additions are: the <code>mlflow<\/code> import, tracing setup (<code>set_trace_processors([])<\/code> and <code>mlflow.openai.autolog()<\/code>), the experiment from an environment variable, and the <code>@mlflow.trace<\/code> decorator on <code>run_agent()<\/code>.<\/p>", "changes": "<strong>What changed:<\/strong><ul><li><code>set_trace_processors([])<\/code>: Stops the Agents SDK from sending its own traces to OpenAI<\/li><li><code>mlflow.openai.autolog()<\/code>: Records each Agents SDK step, such as the model call, as a child span<\/li><li><code>mlflow.set_experiment()<\/code>: Targets the experiment created by the DAB, using the ID injected via environment variable<\/li><li><code>@mlflow.trace<\/code>: Wraps every call in a root span that captures inputs, outputs, and timing<\/li><li><code>mlflow.update_current_trace(tags=...)<\/code>: Tags each trace with the agent and model service, so you can filter traces later<\/li><\/ul>", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom agents.tracing import set_trace_processors\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Send traces to MLflow instead of OpenAI\nset_trace_processors([])\nmlflow.set_tracking_uri(\"databricks\")\nmlflow.openai.autolog()\n\n# Read environment variables injected by the DAB\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\nmlflow.set_experiment(experiment_id=os.getenv(\"MLFLOW_EXPERIMENT_ID\"))\n\n# Define the agent\nagent = Agent(\n    name=\"Simple Chatbot\",\n    instructions=\"You are a helpful assistant. Answer the user's questions clearly and concisely.\",\n    model=MODEL_SERVICE,\n)\n\n\n@mlflow.trace(name=\"mlflow_chatbot_run\", span_type=\"AGENT\")\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent and return the final reply. The trace records each step.\"\"\"\n    mlflow.update_current_trace(tags={\"agent_name\": agent.name, \"model_service\": MODEL_SERVICE})\n    result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "databricks.yml", "desc": "<p>Replace the contents of <strong>databricks.yml<\/strong> with the version below. This adds an MLflow experiment as a bundle-managed resource and grants the app permission to write to it.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the experiment resource manually on the app Overview page during the deploy step.<\/div>", "changes": "<strong>What changed:<\/strong><ul><li><code>resources.experiments.chatbot_experiment<\/code>: Declares an MLflow experiment that DABs creates on deploy. This is a new resource added to the same app.<\/li><li><code>app-experiment<\/code> resource entry: Grants the app's service principal <code>CAN_EDIT<\/code> on the experiment so it can log traces<\/li><li>The app description updated to reflect the new capability. The bundle name and app name remain the same; redeploying updates the existing app in place.<\/li><\/ul>", "code": "bundle:\n  name: simple_chatbot\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n\ntargets:\n  dev:\n    mode: development\n    default: true"}, {"name": "app.yaml", "desc": "<p>Replace the contents of <strong>app.yaml<\/strong> with the version below. This injects the MLflow experiment ID as an environment variable.<\/p>", "changes": "<strong>What changed:<\/strong><ul><li>The new <code>MLFLOW_EXPERIMENT_ID<\/code> entry uses <code>valueFrom<\/code> to resolve the experiment ID from the <code>app-experiment<\/code> resource declared in <code>databricks.yml<\/code>. The agent reads this at startup to target the correct experiment.<\/li><\/ul>", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}], "E1": [{"name": "agent.py", "desc": "<p>Replace the contents of <strong>agent.py<\/strong> with the final version below. This adds MCP imports, a <code>WorkspaceClient<\/code> for authentication, and the URL of the Databricks-provided SQL MCP <code>system.ai.dbsql<\/code>. <code>run_agent()<\/code> now opens an <code>McpServer<\/code> for each request and passes it to the agent.<\/p>", "changes": "<strong>What changed from Section D:<\/strong><ul><li><code>McpServer<\/code> and <code>WorkspaceClient<\/code> imports added<\/li><li><code>LISTINGS_TABLE<\/code>: Your table's full name, injected by its app resource. The instructions name it, so the agent knows what to query.<\/li><li><code>async with McpServer(...)<\/code>: Connects to <code>system.ai.dbsql<\/code> through Unity Gateway for each request and closes the connection after. Its <code>execute_sql<\/code> tool runs SQL on a SQL warehouse.<\/li><li><code>mcp_servers=[mcp_server]<\/code> on the Agent: The SDK lists the MCP's tools at runtime<\/li><\/ul>", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom agents.tracing import set_trace_processors\nfrom databricks.sdk import WorkspaceClient\nfrom databricks_openai import AsyncDatabricksOpenAI\nfrom databricks_openai.agents import McpServer\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Send traces to MLflow instead of OpenAI\nset_trace_processors([])\nmlflow.set_tracking_uri(\"databricks\")\nmlflow.openai.autolog()\n\n# Read environment variables injected by the DAB\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\nmlflow.set_experiment(experiment_id=os.getenv(\"MLFLOW_EXPERIMENT_ID\"))\n\n# MCP configuration: the table's full name comes from its app resource\nLISTINGS_TABLE = os.getenv(\"LISTINGS_TABLE\")\n\nws = WorkspaceClient()\n\n# The Databricks-provided SQL MCP, reached through Unity Gateway\nDBSQL_URL = f\"{ws.config.host}/ai-gateway/mcp-services/system.ai.dbsql\"\n\nINSTRUCTIONS = (\n    \"You are a helpful assistant. Answer the user's questions clearly and concisely. \"\n    f\"For questions about San Francisco Airbnb listings, query {LISTINGS_TABLE} with the execute_sql tool.\"\n)\n\n\n@mlflow.trace(name=\"tool_calling_chatbot_run\", span_type=\"AGENT\")\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent with MCP tools and return the final reply.\"\"\"\n    mlflow.update_current_trace(tags={\"agent_name\": \"Tool-calling Chatbot\", \"model_service\": MODEL_SERVICE})\n    # Connect to the MCP for this request; leaving the block closes the connection\n    async with McpServer(url=DBSQL_URL, workspace_client=ws, name=\"dbsql\", timeout=60.0) as mcp_server:\n        # Define the agent with MCP tool access\n        agent = Agent(\n            name=\"Simple Chatbot\",\n            instructions=INSTRUCTIONS,\n            model=MODEL_SERVICE,\n            mcp_servers=[mcp_server],\n        )\n        result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "app.yaml", "desc": "<p>Replace the contents of <strong>app.yaml<\/strong> with the final version below. This adds a <code>LISTINGS_TABLE<\/code> environment variable with the full name of your listings table.<\/p>", "changes": "<strong>What changed from Section D:<\/strong><ul><li>The new entry uses <code>valueFrom<\/code> to read the table's full name from the <code>listings-table<\/code> resource, so there is nothing to type in.<\/li><\/ul>", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n  - name: LISTINGS_TABLE\n    valueFrom: listings-table\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}, {"name": "databricks.yml", "desc": "<p>Replace the contents of <strong>databricks.yml<\/strong> with the final version below. This adds a <code>variables<\/code> block for your catalog and the SQL warehouse, and two resources: <code>SELECT<\/code> on your listings table and <code>CAN_USE<\/code> on the warehouse.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the table and SQL warehouse resources manually on the app Overview page during the deploy step.<\/div>", "changes": "<strong>What changed from Section D:<\/strong><ul><li><code>variables<\/code> block: <code>catalog_name<\/code> for your catalog, and <code>warehouse_id<\/code>, which looks up the <code>shared_warehouse<\/code> SQL warehouse by name<\/li><li><code>listings-table<\/code> resource: Grants the app’s service principal <code>SELECT<\/code> on <code>sf_airbnb_listings<\/code><\/li><li><code>sql-warehouse<\/code> resource: Grants <code>CAN_USE<\/code> on the warehouse, where the SQL MCP runs the agent’s queries<\/li><li>No grant is needed for <code>system.ai.dbsql<\/code> itself: all workspace users can already call the Databricks-provided MCPs<\/li><\/ul>", "code": "bundle:\n  name: simple_chatbot\n\nvariables:\n  catalog_name:\n    description: \"Per-user lab catalog, e.g. labuser_matthew_mccoy\"\n    default: # <---- Update with your catalog name\n  warehouse_id:\n    description: \"SQL warehouse the SQL MCP runs queries on\"\n    lookup:\n      warehouse: \"shared_warehouse\"\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n        - name: listings-table\n          uc_securable:\n            securable_full_name: ${var.catalog_name}.agent_deployment.sf_airbnb_listings\n            securable_type: TABLE\n            permission: SELECT\n        - name: sql-warehouse\n          sql_warehouse:\n            id: ${var.warehouse_id}\n            permission: CAN_USE\n\ntargets:\n  dev:\n    mode: development\n    default: true"}]}, STAGE = 0, P = "c1";
  var ORDER = ["agent.py", "server.py", "pyproject.toml", "app.yaml", "databricks.yml", "requirements.txt"];
  var LANG = { py: "python", toml: "toml", yaml: "yaml", yml: "yaml", txt: "text" };
  var PYPROJECT_DESC = '<p>Add <code>"mlflow==3.16.1"</code> to the <code>dependencies</code> list in <strong>pyproject.toml</strong>. Without this dependency, the app cannot import <code>mlflow</code> at runtime and will crash on startup.</p>';
  function buildStages() {
    function toMap(list, key) { var m = {}; list.forEach(function (f) { m[f.name] = f[key]; }); return m; }
    var s1 = toMap(RAW.C1, "code"), s2 = Object.assign({}, s1, toMap(RAW.D1, "code"));
    s2["pyproject.toml"] = s1["pyproject.toml"].replace('    "uv",', '    "uv",\n    "mlflow==3.16.1",');
    var s3 = Object.assign({}, s2, toMap(RAW.E1, "code"));
    var metas = [RAW.C1, RAW.D1, RAW.E1].map(function (list) { var m = {}; list.forEach(function (f) { m[f.name] = { desc: f.desc || "", changes: f.changes || "" }; }); return m; });
    metas[1]["pyproject.toml"] = { desc: PYPROJECT_DESC, changes: "" };
    var stages = [s1, s2, s3].map(function (files, i) { return { files: files, meta: metas[i] }; });
    stages.forEach(function (st, i) {
      var prev = i ? stages[i - 1].files : {};
      st.prev = prev;
      st.changed = ORDER.filter(function (n) { return prev[n] !== st.files[n]; });
    });
    return stages;
  }
  function langOf(name) { return LANG[name.split(".").pop()] || "text"; }
  function esc(s) { return s.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;"); }
  function hlLine(line, lang) {
    var re;
    if (lang === "python") re = /(#.*$)|("""[\s\S]*?"""|"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:from|import|def|class|return|if|elif|else|for|with|as|async|await|None|True|False|and|or|not|in)\b)|(@[\w.]+)|(\b\w+(?=\())|(\b\d+\.?\d*\b)/g;
    else if (lang === "yaml" || lang === "toml") re = /(#.*$)|("(?:[^"\\]|\\.)*")|(\$\{[^}]+\})|(^\s*-?\s*)([\w][\w\-.]*)(?=\s*[:=])|(\[[\w.\-]+\])/g;
    else return esc(line);
    var out = "", last = 0, m;
    while ((m = re.exec(line))) {
      out += esc(line.slice(last, m.index));
      if (lang === "python") {
        var cls = m[1] ? "c" : m[2] ? "s" : m[3] ? "k" : m[4] ? "d" : m[5] ? "f" : "n";
        out += '<span class="t-' + cls + '">' + esc(m[0]) + "</span>";
      } else if (m[1]) out += '<span class="t-c">' + esc(m[0]) + "</span>";
      else if (m[2]) out += '<span class="t-s">' + esc(m[0]) + "</span>";
      else if (m[3]) out += '<span class="t-d">' + esc(m[0]) + "</span>";
      else if (m[5]) out += esc(m[4]) + '<span class="t-y">' + esc(m[5]) + "</span>";
      else out += '<span class="t-k">' + esc(m[0]) + "</span>";
      last = m.index + m[0].length;
      if (m[0] === "") re.lastIndex++;
    }
    return out + esc(line.slice(last));
  }
  function addedSet(a, b) {
    a = a ? a.split("\n") : []; b = b.split("\n");
    var n = a.length, m = b.length, dp = [], i, j, set = {}, k = 0;
    for (i = 0; i <= n; i++) dp.push(new Array(m + 1).fill(0));
    for (i = n - 1; i >= 0; i--) for (j = m - 1; j >= 0; j--) dp[i][j] = a[i] === b[j] ? dp[i + 1][j + 1] + 1 : Math.max(dp[i + 1][j], dp[i][j + 1]);
    i = 0; j = 0;
    while (j < m) {
      if (i < n && a[i] === b[j]) { i++; j++; }
      else if (i < n && dp[i + 1][j] >= dp[i][j + 1]) i++;
      else set[j++] = 1;
    }
    return set;
  }
  var COPY_ICON = '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>';
  function copyText(text, btn) {
    var t = document.createElement("textarea"); t.value = text; t.style.position = "fixed"; t.style.opacity = "0"; document.body.appendChild(t); t.select();
    var ok = false; try { ok = document.execCommand("copy"); } catch (e) {} document.body.removeChild(t);
    if (!ok && navigator.clipboard) { navigator.clipboard.writeText(text); ok = true; }
    if (ok) { btn.classList.add("ok"); btn.innerHTML = "Copied"; setTimeout(function () { btn.classList.remove("ok"); btn.innerHTML = COPY_ICON + "Copy"; }, 1500); }
  }
  var S = buildStages(), st = S[STAGE];
  var CONTS = [
    { id: "app", x: 215, y: 84, w: 505, h: 296, label: "DATABRICKS APP" },
    { id: "gw", x: 790, y: 40, w: 270, h: STAGE < 2 ? 180 : 340, label: "UNITY GATEWAY" }
  ];
  var NODES = [
    { id: "browser", x: 14, y: 170, w: 122, h: 84, t: "Browser", s: "chat.html", from: 0 },
    { id: "server", x: 240, y: 170, w: 175, h: 84, t: "server.py", s: "FastAPI", from: 0, mono: 1 },
    { id: "agent", x: 522, y: 170, w: 175, h: 84, t: "agent.py", s: "OpenAI Agents SDK", from: 0, mono: 1 },
    { id: "runtime", x: 240, y: 292, w: 175, h: 62, t: "uv run start-server", s: "", from: 0, mono: 1, small: 1 },
    { id: "model", x: 815, y: 104, w: 220, h: 84, t: "agent-model", s: "Model service", from: 0, perm: "EXECUTE", mono: 1 },
    { id: "mcp", x: 815, y: 264, w: 220, h: 84, t: "system.ai.dbsql", s: "MCP server", from: 2, mono: 1 },
    { id: "warehouse", x: 1100, y: 264, w: 200, h: 84, t: "shared_warehouse", s: "SQL warehouse", from: 2, perm: "CAN_USE", mono: 1 },
    { id: "table", x: 1100, y: 416, w: 200, h: 84, t: "sf_airbnb_listings", s: "Unity Catalog table", from: 2, perm: "SELECT", mono: 1 },
    { id: "exp", x: 479, y: 420, w: 262, h: 84, t: "MLflow experiment", s: ["", "Traces", "Traces + tool spans"][STAGE], from: 1, perm: "CAN_EDIT" }
  ];
  var EDGES = [
    { a: "browser", b: "server", l: "/invocations", from: 0 },
    { a: "server", b: "agent", l: "run_agent()", from: 0 },
    { a: "agent", b: "model", l: "Responses API", from: 0, end: 1 },
    { a: "agent", b: "mcp", l: "execute_sql", from: 2, end: 1 },
    { a: "mcp", b: "warehouse", l: "SQL", from: 2 },
    { a: "warehouse", b: "table", l: "", from: 2, down: 1 },
    { a: "agent", b: "exp", l: "", from: 1, down: 1 }
  ];
  // [file, line prefix, include the indented block that follows]
  var MAP = {
    browser: [["server.py", "CHAT_UI"], ["server.py", "@app.get"], ["server.py", "async def root"], ["server.py", "return CHAT_UI"]],
    server: [["server.py", "import uvicorn"], ["server.py", "from fastapi"], ["server.py", "from pydantic"], ["server.py", "from agent import"], ["server.py", "app = FastAPI"],
      ["server.py", "class ChatRequest", 1], ["server.py", "@app.post"], ["server.py", "async def invocations", 1], ["server.py", "def main", 1]],
    agent: [["agent.py", "from agents import"], ["agent.py", "# Define"], ["agent.py", "agent = Agent(", 1], ["agent.py", "async def run_agent"], ["agent.py", "result = await Runner.run"], ["agent.py", "return"], ["agent.py", "final_output = result"]],
    runtime: [["app.yaml", "command:"], ["pyproject.toml", "dependencies = [", 1], ["pyproject.toml", "[tool.hatch"], ["pyproject.toml", "include ="], ["pyproject.toml", "[project.scripts]"], ["pyproject.toml", "start-server"], ["requirements.txt", "uv"]],
    app: [["databricks.yml", "bundle:", 1], ["databricks.yml", "apps:"], ["databricks.yml", "simple_chatbot:"], ["databricks.yml", "name: agent-"], ["databricks.yml", "description:"], ["databricks.yml", "source_code_path:"], ["databricks.yml", "resources:"], ["databricks.yml", "targets:", 1]],
    gw: [["agent.py", "set_default_openai_client"], ["agent.py", "DBSQL_URL ="], ["agent.py", "# The Databricks-provided"]],
    model: [["agent.py", "from databricks_openai import AsyncDatabricksOpenAI"], ["agent.py", "# Point the"], ["agent.py", "set_default_openai_client"], ["agent.py", "set_default_openai_api"], ["agent.py", "# Read the model"],
      ["agent.py", "MODEL_SERVICE ="], ["agent.py", "model=MODEL_SERVICE"], ["app.yaml", "- name: MODEL_SERVICE", 1], ["databricks.yml", "- name: model-service", 1]],
    exp: [["agent.py", "from agents.tracing"], ["agent.py", "import mlflow"], ["agent.py", "# Send traces"], ["agent.py", "set_trace_processors("], ["agent.py", "mlflow.set_tracking_uri"], ["agent.py", "mlflow.openai.autolog"], ["agent.py", "mlflow.set_experiment"],
      ["agent.py", "@mlflow.trace"], ["agent.py", "mlflow.update_current_trace"], ["app.yaml", "- name: MLFLOW_EXPERIMENT_ID", 1], ["databricks.yml", "experiments:", 1], ["databricks.yml", "- name: app-experiment", 1], ["pyproject.toml", "\"mlflow=="]],
    mcp: [["agent.py", "from databricks_openai.agents"], ["agent.py", "from databricks.sdk"], ["agent.py", "ws = WorkspaceClient"], ["agent.py", "# The Databricks-provided"], ["agent.py", "DBSQL_URL ="], ["agent.py", "# Connect to the MCP"], ["agent.py", "async with McpServer("], ["agent.py", "mcp_servers="]],
    warehouse: [["databricks.yml", "warehouse_id:", 1], ["databricks.yml", "- name: sql-warehouse", 1]],
    table: [["agent.py", "# MCP configuration"], ["agent.py", "LISTINGS_TABLE ="], ["agent.py", "f\"For questions"], ["app.yaml", "- name: LISTINGS_TABLE", 1], ["databricks.yml", "catalog_name:", 1], ["databricks.yml", "- name: listings-table", 1]]
  };
  var sel = null, file = st.changed[0];
  function $(id) { return document.getElementById(P + "-" + id); }
  function ind(l) { return l.length - l.replace(/^\s+/, "").length; }
  function hits(id, name) {
    var lines = st.files[name].split("\n"), set = {};
    (MAP[id] || []).forEach(function (m) {
      if (m[0] !== name) return;
      lines.forEach(function (l, i) {
        if (l.trim().indexOf(m[1]) !== 0) return;
        set[i] = 1;
        if (!m[2]) return;
        var base = ind(l), j = i + 1;
        while (j < lines.length && (!lines[j].trim() || ind(lines[j]) > base)) { if (lines[j].trim()) set[j] = 1; j++; }
        if (j < lines.length && /^[\)\]]/.test(lines[j].trim()) && ind(lines[j]) === base) set[j] = 1;
      });
    });
    return set;
  }
  function count(o) { return Object.keys(o).length; }
  function N(id) { return NODES.filter(function (n) { return n.id === id; })[0]; }
  function vis(o) { return o.from <= STAGE; }
  function isNew(o) { return STAGE > 0 && o.from === STAGE; }
  function draw() {
    var h = '<defs><marker id="' + P + '-ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#9AABB1"/></marker>' +
      '<marker id="' + P + '-arn" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#FF5F46"/></marker></defs>';
    CONTS.forEach(function (c) {
      h += '<g class="cont' + (sel === c.id ? " sel" : "") + '"><rect class="box" x="' + c.x + '" y="' + c.y + '" width="' + c.w + '" height="' + c.h + '" rx="16"/>' +
        '<g class="hdr" data-id="' + c.id + '"><rect x="' + c.x + '" y="' + c.y + '" width="' + c.w + '" height="40" rx="16" fill="transparent"/><text x="' + (c.x + 22) + '" y="' + (c.y + 27) + '">' + c.label + "</text></g></g>";
    });
    EDGES.forEach(function (e) {
      if (!vis(e)) return;
      var a = N(e.a), b = N(e.b), d, lx, x1, y1, x2, y2;
      if (e.down) { x1 = a.x + a.w / 2; y1 = a.y + a.h; d = "M" + x1 + "," + y1 + " L" + x1 + "," + (b.y - 2); }
      else {
        x1 = a.x + a.w; y1 = a.y + a.h / 2; x2 = b.x; y2 = b.y + b.h / 2; lx = (x1 + x2) / 2;
        d = "M" + x1 + "," + y1 + " C" + lx + "," + y1 + " " + lx + "," + y2 + " " + (x2 - 2) + "," + y2;
      }
      var lab = !e.l ? "" : e.end ? '<text x="' + (b.x + 4) + '" y="' + (b.y - 9) + '">' + e.l + "</text>" : '<text x="' + lx + '" y="' + (y1 - 10) + '" text-anchor="middle">' + e.l + "</text>";
      h += '<g class="ed' + (isNew(e) ? " new" : "") + '"><path d="' + d + '" marker-end="url(#' + P + (isNew(e) ? "-arn" : "-ar") + ')"/>' + lab + "</g>";
    });
    NODES.forEach(function (n) {
      if (!vis(n)) return;
      var cy = n.y + n.h / 2;
      h += '<g class="nd' + (n.mono ? " mono" : "") + (n.small ? " small" : "") + (isNew(n) ? " new" : "") + (sel === n.id ? " sel" : "") + '" data-id="' + n.id + '">' +
        '<rect class="box" x="' + n.x + '" y="' + n.y + '" width="' + n.w + '" height="' + n.h + '" rx="10"/>' +
        '<text class="t" x="' + (n.x + n.w / 2) + '" y="' + (n.s ? cy - 4 : cy + 5) + '" text-anchor="middle">' + n.t + "</text>" +
        (n.s ? '<text class="s" x="' + (n.x + n.w / 2) + '" y="' + (cy + 17) + '" text-anchor="middle">' + n.s + "</text>" : "");
      if (n.perm) { var pw = n.perm.length * 7.6 + 16; h += '<g class="pill"><rect x="' + (n.x + n.w - pw - 10) + '" y="' + (n.y - 10) + '" width="' + pw + '" height="20" rx="10"/><text x="' + (n.x + n.w - 10 - pw / 2) + '" y="' + (n.y + 4) + '" text-anchor="middle">' + n.perm + "</text></g>"; }
      h += "</g>";
    });
    var svg = $("svg");
    svg.setAttribute("viewBox", "0 30 " + (STAGE < 2 ? 1080 : 1320) + " " + (STAGE < 1 ? 360 : 486));
    svg.innerHTML = h;
    Array.prototype.forEach.call(svg.querySelectorAll("[data-id]"), function (g) { g.addEventListener("click", function () { pick(g.getAttribute("data-id")); }); });
  }
  function pick(id) {
    sel = sel === id ? null : id;
    if (sel && !count(hits(sel, file))) {
      var withHits = ORDER.filter(function (n) { return count(hits(sel, n)); });
      file = withHits.filter(function (n) { return st.changed.indexOf(n) >= 0; })[0] || withHits[0] || file;
    }
    render();
  }
  function render() {
    draw();
    var t = $("tabs");
    t.innerHTML = ORDER.map(function (n) {
      var ch = st.changed.indexOf(n) >= 0, k = sel ? count(hits(sel, n)) : 0;
      return '<div class="tab' + (n === file ? " on" : "") + (ch ? "" : " same") + '" data-f="' + n + '">' + n + (STAGE && ch ? '<span class="d"></span>' : "") + (k ? '<span class="n">' + k + "</span>" : "") + "</div>";
    }).join("") + '<span class="grow"></span><button class="cp" id="' + P + '-cp">' + COPY_ICON + "Copy</button>";
    Array.prototype.forEach.call(t.querySelectorAll(".tab"), function (el) { el.addEventListener("click", function () { file = el.getAttribute("data-f"); render(); }); });
    var meta = st.changed.indexOf(file) >= 0 && st.meta[file];
    $("desc").innerHTML = meta ? '<div class="desc">' + meta.desc + (meta.changes ? '<div class="chg">' + meta.changes + "</div>" : "") + "</div>" : "";
    $("split").className = "split" + (meta ? "" : " full");
    var hs = sel ? hits(sel, file) : null, add = STAGE && st.changed.indexOf(file) >= 0 ? addedSet(st.prev[file], st.files[file]) : {};
    var c = $("code");
    c.className = "code" + (hs && count(hs) ? " focus" : "");
    c.innerHTML = st.files[file].split("\n").map(function (l, i) {
      return '<div class="ln' + (add[i] ? " add" : "") + (hs && hs[i] ? " hit" : "") + '"><span class="g">' + (add[i] ? "+" : i + 1) + "</span><span>" + (hlLine(l, langOf(file)) || " ") + "</span></div>";
    }).join("");
    c.style.maxHeight = ""; /* height now follows the description panel via CSS grid stretch */
    var first = c.querySelector(".hit"); if (first) c.scrollTop = first.offsetTop - c.offsetTop - 40;
    var b = $("cp"); b.onclick = function () { copyText(st.files[file], b); };
  }
  render();
})();
</script>

Run the next cell to generate a unique identifier if you are deploying the app via the UI. That is, you are _not_ working in a git folder.

In [0]:
user_email = spark.sql("SELECT current_user()").first()[0]
uniqueIdentifier1 = user_email.split("@")[0].replace('labuser','').replace("_","-").replace(".","-")
print(f"agent-{uniqueIdentifier1}-dev")

### C2. Deploy the App

There are two ways to deploy, depending on whether your workspace files are inside a Git folder. Select a card below.

<br></br>

<div style="width: 100%; font-family: sans-serif;">

<style>
.c2d-row { padding: 4px 6px; box-sizing: border-box; display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
.c2d-box {
  display: block; background: #fff; border: 2px solid #C6DCEA; box-sizing: border-box; border-radius: 12px;
  padding: 16px 20px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);
  cursor: pointer; user-select: none; text-align: left;
  transition: transform 0.12s, box-shadow 0.15s, border-color 0.15s;
}
.c2d-box::before { content: ""; display: block; height: 4px; border-radius: 2px; background: var(--pc); margin: -2px 0 14px; }
.c2d-box:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(27,49,57,.10); }
.c2d-box.active { border-color: var(--pc); box-shadow: 0 6px 16px rgba(27,49,57,.10); }
.c2d-pill {
  display: inline-block; background: var(--pc); color: #fff; font-size: 10.5pt; font-weight: 700;
  letter-spacing: 0.06em; border-radius: 999px; padding: 3px 12px; pointer-events: none;
}
.c2d-title { display: block; font-size: 16pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 10px; pointer-events: none; }
.c2d-subtitle { display: block; font-size: 12.5pt; font-weight: 500; color: #5A6F77; line-height: 1.3; margin-top: 3px; pointer-events: none; }
.c2d-detail-wrap {
  padding: 0 6px 8px; box-sizing: border-box; overflow: hidden; max-height: 0; opacity: 0; margin-top: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
}
.c2d-detail-wrap.open { max-height: 2400px; opacity: 1; margin-top: 16px; }
.c2d-detail-card {
  background: #fff; border: 1.5px solid #C6DCEA; border-radius: 12px;
  padding: 18px 22px 20px; box-sizing: border-box; box-shadow: 0 2px 8px rgba(27,49,57,.07);
}
.c2d-accent { height: 4px; border-radius: 2px; margin: -2px 0 14px; }
.c2d-steps-block { font-size: 14.5pt; line-height: 1.6; color: #0b2026; margin-bottom: 14px; }
.c2d-steps-block ol { margin: 8px 0 0 0; padding-left: 22px; }
.c2d-steps-block li { margin-bottom: 8px; }
.c2d-steps-block ul { margin: 4px 0 8px 20px; list-style: disc; }
.c2d-info-block {
  background: #F9F7F4; border-left: 4px solid; border-radius: 0 6px 6px 0;
  padding: 12px 14px; font-size: 14pt; line-height: 1.5; color: #0b2026;
}
</style>

<div>
  <div class="c2d-row">
    <span class="c2d-box active" data-id="0" onclick="c2dSelect(0)" style="--pc:#5A6F77;">
      <span class="c2d-pill">DABS UI</span>
      <span class="c2d-title">Git folder</span>
      <span class="c2d-subtitle">Deploys from databricks.yml</span>
    </span>
    <span class="c2d-box" data-id="1" onclick="c2dSelect(1)" style="--pc:#FF5F46;">
      <span class="c2d-pill">APPS UI</span>
      <span class="c2d-title">No Git folder</span>
      <span class="c2d-subtitle">Add resources by hand</span>
    </span>
  </div>

  <div class="c2d-detail-wrap" id="c2d-detail-wrap">
    <div class="c2d-detail-card" id="c2d-detail-card"></div>
  </div>

</div>

</div>

<script>
var C2D_CARDS = [
  {
    title: "DABs UI (Git folder)",
    color: "#5A6F77",
    bg: "rgba(90,111,119,0.10)",
    infoBg: "rgba(90,111,119,0.10)",
    content: `
      <ol>
        <li>Navigate inside the project folder and click on any file (e.g., <strong>databricks.yml</strong>). A rocket icon will appear. Click it.</li>
        <li>Click <strong>Deploy</strong> in the <strong>Deployments</strong> section. This DAB is configured to deploy to <strong>dev</strong> only.</li>
        <li>Once validation clears and the deployment summary appears, click <strong>Deploy</strong> again to confirm.</li>
        <li>Under <strong>Bundle resource</strong>, click the app link (e.g., <strong>agent-userid-dev</strong>) to open the <strong>Apps</strong> page.</li>
        <li>Click <strong>Start</strong> to provision compute. After a few minutes, click <strong>Deploy</strong> to push the source code to the running app.
          <ul>
            <li>Navigate back to the source folder of the app <strong>simple-agent</strong> and click the three vertical dots next to the folder name. Select <strong>Copy URL/path</strong> and select <strong>full path</strong>.</li>
            <li>Paste this in the <strong>source code path</strong> textbox in the app's <strong>Create deployment</strong> menu.</li>
            <li>Click <strong>Deploy</strong>.</li>
          </ul>
        </li>
        <li>Once the app shows <strong>Running</strong> status, click the endpoint to open the chat UI and verify the agent responds by sending a query like <em>"Please tell me about Apache Spark."</em></li>
      </ol>
    `,
    info: '<strong>Requires a Git folder.</strong> The DABs UI automates resource creation and permission grants from your <code>databricks.yml</code> file.'
  },
  {
    title: "Apps UI (no Git folder)",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    infoBg: "rgba(255,95,70,0.10)",
    content: `
      <ol>
        <li>In the workspace sidebar, click <strong>+ New</strong> and select <strong>App</strong>.</li>
        <li>Click <strong>Create a custom app</strong> in the <strong>Apps</strong> landing page.</li>
        <li>Give the app a name (e.g., <strong>agent-uniqueIdentifier-dev</strong> as indicated in the previous cell) and click <strong>Create app</strong>.</li>
        <li>Wait for the compute to provision.
        </li>
        <li>Configure the app resources manually: add a <strong>model service</strong> resource and map it to <code>workspace.agent_deployment.agent-model</code> with <strong>Can execute</strong> permission.
          <ul>
            <li>Click on <strong>(⋮)</strong> icon then on  <strong>Edit</strong> at the top right.</li>
            <li>Click <strong>Resources</strong> at the top of the setup wizard.</li>
            <li>Click <strong>+ Add resource</strong> and select <strong>Model</strong>.</li>
            <li>In the dropdown menu search for <strong>agent-model</strong> and select <code>workspace.agent_deployment.agent-model</code>. Make sure <strong>Permission</strong> is set to <strong>Can execute</strong>. Make sure <strong>Resource key</strong> is set to <strong>model-service</strong>.</li>
            <li>Click <strong>Save</strong>.</li>
          </ul>
        </li>
        <li>On the app's Overview page, click <strong>Deploy</strong> and set the <strong>Source path</strong> to the workspace folder containing your project files.
          <ul>
            <li>Navigate back to the source folder of the app <strong>simple-agent</strong> and click the three vertical dots next to the folder name. Select <strong>Copy URL/path</strong> and select <strong>full path</strong>.</li>
            <li>Paste this in the <strong>source code path</strong> textbox in the app's <strong>Create deployment</strong> menu.</li>
            <li>Click <strong>Deploy</strong>.</li>
          </ul>
        </li>
        <li>Click the endpoint URL to open the chat UI and verify the agent responds by sending a query like <em>"Please tell me about Apache Spark."</em></li>
      </ol>
    `,
    info: '<strong>No Git folder required.</strong> You configure app resources manually on the Overview page.'
  }
];

var c2dCurrent = null;

function c2dSelect(id) {
  var wrap = document.getElementById('c2d-detail-wrap');
  var card = document.getElementById('c2d-detail-card');
  var c = C2D_CARDS[id];

  document.querySelectorAll('.c2d-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (c2dCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.c2d-box').forEach(function(b) {
      b.classList.remove('active');
    });
    c2dCurrent = null;
    return;
  }

  c2dCurrent = id;

  var label = c.title.split(' (')[0].toUpperCase();
  var sub = c.title.replace(/^.*\((.*)\)$/, '$1'); sub = sub.charAt(0).toUpperCase() + sub.slice(1);
  card.innerHTML =
    '<div class="c2d-accent" style="background:' + c.color + ';"></div>'
    + '<span class="c2d-pill" style="--pc:' + c.color + ';">' + label + '</span>'
    + '<div style="font-size:18pt;font-weight:700;margin:10px 0 12px;color:#0b2026;">' + sub + '</div>'
    + '<div class="c2d-steps-block">'
    + c.content
    + '</div>'
    + '<div class="c2d-info-block" style="border-color:' + c.color + ';">'
    + c.info
    + '</div>';

  wrap.classList.add('open');
}

c2dSelect(0);
</script>

## D. Add MLflow Tracing

With the basic agent running, the next step is to add observability. You will update four files to wire in MLflow tracing so every conversation produces a structured trace.

### D1. Updated Files

Tracing adds an MLflow experiment and changes four files. Green lines are new. Select a component in the diagram to highlight its code.

<style>
#d1-arch { width:100%; margin:12px 0 0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; color:#0b2026; }
#d1-arch .dia { margin:0 8px; background:#fff; border:1px solid #DCE0E2; border-radius:14px; padding:12px; box-shadow:0 4px 18px rgba(27,49,57,.06); }
#d1-arch svg { display:block; width:100%; height:auto; }
#d1-arch svg text { font-family:inherit; fill:#0b2026; }
#d1-arch .cont rect.box { fill:#F9F7F4; stroke:#DCE0E2; stroke-width:1.5; }
#d1-arch .cont .hdr { cursor:pointer; }
#d1-arch .cont .hdr text { font-size:13px; font-weight:700; letter-spacing:1.4px; fill:#618794; }
#d1-arch .cont .hdr:hover text, #d1-arch .cont.sel .hdr text { fill:#FF5F46; }
#d1-arch .nd { cursor:pointer; }
#d1-arch .nd rect.box { fill:#fff; stroke:#C9D0D3; stroke-width:1.5; transition:fill .15s, stroke .15s; }
#d1-arch .nd:hover rect.box { stroke:#1B5162; }
#d1-arch .nd .t { font-size:16px; font-weight:700; }
#d1-arch .nd .s { font-size:13px; fill:#618794; }
#d1-arch .nd.mono .t { font-family:ui-monospace,Menlo,Consolas,monospace; font-size:15px; }
#d1-arch .nd.small .t { font-size:13.5px; }
#d1-arch .nd.new rect.box { stroke:#FF5F46; stroke-width:2.2; }
#d1-arch .nd.sel rect.box { fill:#1B5162; stroke:#1B5162; }
#d1-arch .nd.sel .t, #d1-arch .nd.sel .s { fill:#fff; }
#d1-arch .pill rect { fill:#fff; stroke:#1B5162; stroke-width:1.2; }
#d1-arch .pill text { font-size:10.5px; font-weight:700; letter-spacing:.6px; fill:#1B5162; font-family:ui-monospace,Menlo,Consolas,monospace; }
#d1-arch .ed path { fill:none; stroke:#9AABB1; stroke-width:2; }
#d1-arch .ed.new path { stroke:#FF5F46; }
#d1-arch .ed text { font-size:12px; fill:#618794; font-family:ui-monospace,Menlo,Consolas,monospace; }
#d1-arch .ed.new text { fill:#FF5F46; }
#d1-arch .panel { margin-top:14px; border:1px solid #DCE0E2; border-radius:12px; overflow:hidden; background:#fff; }
#d1-arch .tabs { display:flex; flex-wrap:wrap; align-items:flex-end; gap:2px; background:#F9F7F4; border-bottom:1px solid #DCE0E2; padding:6px 8px 0; }
#d1-arch .tab { padding:8px 13px; font-size:13px; font-weight:600; color:#618794; border-radius:7px 7px 0 0; cursor:pointer; display:flex; gap:6px; align-items:center; font-family:ui-monospace,Menlo,Consolas,monospace; }
#d1-arch .tab.same { color:#AEB8BC; }
#d1-arch .tab.on { background:#fff; color:#0b2026; box-shadow:0 -1px 0 #DCE0E2, 1px 0 0 #DCE0E2, -1px 0 0 #DCE0E2; }
#d1-arch .tab .d { width:7px; height:7px; border-radius:50%; background:#FF5F46; }
#d1-arch .tab .n { font-size:10.5px; font-weight:700; color:#fff; background:#1B5162; border-radius:999px; padding:1px 6px; }
#d1-arch .grow { flex:1; }
#d1-arch .cp { display:inline-flex; flex:none; white-space:nowrap; min-width:84px; justify-content:center; align-items:center; gap:6px; margin-bottom:6px; border:1px solid #1B3139; background:#1B3139; color:#fff; padding:6px 12px; border-radius:6px; font-size:13px; font-weight:600; cursor:pointer; }
#d1-arch .cp.ok { background:#00A972; border-color:#00A972; }
#d1-arch .split { display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; align-items:start; }
#d1-arch .split.full { grid-template-columns:1fr; }
#d1-arch .split.full .side { display:none; }
#d1-arch .side { border-right:1px solid #EEEDE9; background:#fff; min-width:0; overflow-wrap:anywhere; }
#d1-arch .side code { overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
@media (min-width: 821px) { #d1-arch .split:not(.full) { align-items:stretch; grid-template-rows:minmax(420px, auto); } #d1-arch .split:not(.full) .code { height:0; min-height:100%; max-height:none; } }
@media (max-width: 820px) { #d1-arch .split { grid-template-columns:1fr; } #d1-arch .side { border-right:none; border-bottom:1px solid #EEEDE9; max-height:none; } }
#d1-arch .desc { padding:16px 20px 6px; }
#d1-arch .desc p { font-size:13pt; line-height:1.6; color:#333; margin:0 0 12px 0; }
#d1-arch .desc ul { margin:6px 0 10px 18px; padding-left:0; font-size:13pt; line-height:1.6; color:#333; }
#d1-arch .desc li { margin-bottom:4px; }
#d1-arch .chg { border-left:4px solid #1B5162; background:rgba(27,81,98,.06); border-radius:0 6px 6px 0; padding:10px 14px; margin:0 0 12px 0; font-size:12.5pt; line-height:1.55; }
#d1-arch .chg ul { margin:6px 0 0 20px; }
#d1-arch .code { font-size:13px; line-height:1.65; padding:10px 0; max-height:480px; overflow-y:auto; overflow-x:hidden; background:#fcfcfb; font-family:ui-monospace,Menlo,Consolas,monospace; color:#383a42; }
#d1-arch .ln { display:flex; white-space:pre-wrap; overflow-wrap:anywhere; border-left:3px solid transparent; transition:opacity .2s, background .2s; }
#d1-arch .ln > span:last-child { flex:1; min-width:0; padding-right:16px; }
#d1-arch .ln .g { width:40px; flex:none; text-align:right; padding-right:14px; color:#C4CBCE; user-select:none; }
#d1-arch .ln.add { background:rgba(0,169,114,.13); border-left-color:#00A972; }
#d1-arch .ln.add .g { color:#00A972; font-weight:700; }
#d1-arch .code.focus .ln { opacity:.32; }
#d1-arch .code.focus .ln.hit { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
#d1-arch .code.focus .ln.hit.add { background:rgba(0,169,114,.16); border-left-color:#00A972; }
#d1-arch .t-k { color:#a626a4 } #d1-arch .t-f { color:#4078f2 } #d1-arch .t-s { color:#50a14f } #d1-arch .t-n { color:#986801 } #d1-arch .t-c { color:#a0a1a7; font-style:italic } #d1-arch .t-d { color:#c18401 } #d1-arch .t-y { color:#e45649 }
</style>
<div id="d1-arch">
  <div class="dia"><svg id="d1-svg"></svg></div>
  <div class="panel"><div class="tabs" id="d1-tabs"></div><div class="split" id="d1-split"><div class="side" id="d1-desc"></div><div class="code" id="d1-code"></div></div></div>
</div>
<script>
(function () {
  var RAW = {"C1": [{"name": "agent.py", "desc": "<p>This file defines the agent and an async function to run it. The key integration point is <code>set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))<\/code>, which routes all model calls through Databricks instead of the public OpenAI service.<\/p><p><strong>Key components:<\/strong><\/p><ul><li><code>Agent<\/code> and <code>Runner<\/code>: Define and execute the agent workflow from the OpenAI Agents SDK<\/li><li><code>AsyncDatabricksOpenAI<\/code>: An async client that calls Unity Gateway model services using the OpenAI-compatible interface<\/li><li><code>set_default_openai_api(\"responses\")<\/code>: Sends requests through the Responses API, which supports tool calls with reasoning models<\/li><li><code>MODEL_SERVICE<\/code>: The model service name, read from an environment variable injected by the DAB at deploy time<\/li><\/ul>", "changes": "", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport os\n\n# Read the model service name injected by Databricks Apps\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\n\n# Point the OpenAI Agents SDK at Databricks instead of OpenAI\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Define the agent\nagent = Agent(\n    name=\"Simple Chatbot\",\n    instructions=\"You are a helpful assistant. Answer the user's questions clearly and concisely.\",\n    model=MODEL_SERVICE,\n)\n\n\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent with a list of messages and return the final reply.\"\"\"\n    result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "server.py", "desc": "<p>This FastAPI server exposes two endpoints: <code>GET /<\/code> returns a chat HTML page, and <code>POST /invocations<\/code> forwards conversation messages to the agent and returns the reply. It listens on the port Databricks Apps sets in <code>DATABRICKS_APP_PORT<\/code>.<\/p><p>This demo uses a small hand-written server so each piece is visible. For production agents, the <a href=\"https://github.com/databricks/app-templates/tree/main/agent-openai-agents-sdk\" target=\"_blank\" style=\"color: #1976d2; text-decoration: underline;\">agent app template<\/a> uses MLflow <code>AgentServer<\/code>, which adds the standard Responses API, streaming, and a built-in chat UI.<\/p>", "changes": "", "code": "import os\nimport uvicorn\nfrom pathlib import Path\n\nfrom fastapi import FastAPI\nfrom fastapi.responses import HTMLResponse\nfrom pydantic import BaseModel\n\nfrom agent import run_agent\n\napp = FastAPI()\n\nCHAT_UI = Path(__file__).parent / \"chat.html\"\n\n\nclass ChatRequest(BaseModel):\n    input: list[dict]\n\n\n@app.get(\"/\", response_class=HTMLResponse)\nasync def root():\n    return CHAT_UI.read_text()\n\n\n@app.post(\"/invocations\")\nasync def invocations(request: ChatRequest):\n    reply = await run_agent(request.input)\n    return {\"output\": reply}\n\n\ndef main():\n    uvicorn.run(\"server:app\", host=\"0.0.0.0\", port=int(os.getenv(\"DATABRICKS_APP_PORT\", 8000)))\n\n\nif __name__ == \"__main__\":\n    main()"}, {"name": "pyproject.toml", "desc": "<p>Standard Python project metadata and dependency declaration. The <code>[project.scripts]<\/code> entry creates the <code>start-server<\/code> CLI command referenced by <code>app.yaml<\/code>.<\/p>", "changes": "", "code": "[project]\nname = \"simple-chatbot\"\nversion = \"0.1.0\"\nrequires-python = \">=3.11\"\ndependencies = [\n    \"openai-agents>=0.6.0\",\n    \"mcp<2\",\n    \"databricks-openai==0.17.1\",\n    \"databricks-sdk==0.145.0\",\n    \"fastapi>=0.115.0\",\n    \"uvicorn>=0.34.0\",\n    \"uv\",\n]\n\n[build-system]\nrequires = [\"hatchling\"]\nbuild-backend = \"hatchling.build\"\n\n[tool.hatch.build.targets.wheel]\ninclude = [\"agent.py\", \"server.py\"]\n\n[project.scripts]\nstart-server = \"server:main\""}, {"name": "app.yaml", "desc": "<p>The Databricks App manifest. It declares which environment variables to inject (resolved from DAB resources) and the startup command.<\/p>", "changes": "", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}, {"name": "databricks.yml", "desc": "<p>The Declarative Automation Bundle (DAB) manifest. It names the app, grants it <code>EXECUTE<\/code> on the course model service <code>workspace.agent_deployment.agent-model<\/code>, and targets the <code>dev<\/code> deployment mode. The app name uses bundle substitutions (<code>${workspace.current_user.id}<\/code>, <code>${bundle.target}<\/code>) so each user gets a unique app.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file is only needed for DABs-based deployment. If you are deploying via the Apps UI instead, skip this file and configure the app resources manually on the app Overview page.<\/div>", "changes": "", "code": "bundle:\n  name: simple_chatbot\n\nresources:\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n\ntargets:\n  dev:\n    mode: development\n    default: true"}, {"name": "requirements.txt", "desc": "<p>Databricks Apps automatically runs <code>pip install -r requirements.txt<\/code> during deployment. Installing <code>uv<\/code> here ensures the <code>uv run start-server<\/code> command in <code>app.yaml<\/code> works correctly.<\/p>", "changes": "", "code": "uv"}], "D1": [{"name": "agent.py", "desc": "<p>Replace the contents of <strong>agent.py<\/strong> with the version below. The additions are: the <code>mlflow<\/code> import, tracing setup (<code>set_trace_processors([])<\/code> and <code>mlflow.openai.autolog()<\/code>), the experiment from an environment variable, and the <code>@mlflow.trace<\/code> decorator on <code>run_agent()<\/code>.<\/p>", "changes": "<strong>What changed:<\/strong><ul><li><code>set_trace_processors([])<\/code>: Stops the Agents SDK from sending its own traces to OpenAI<\/li><li><code>mlflow.openai.autolog()<\/code>: Records each Agents SDK step, such as the model call, as a child span<\/li><li><code>mlflow.set_experiment()<\/code>: Targets the experiment created by the DAB, using the ID injected via environment variable<\/li><li><code>@mlflow.trace<\/code>: Wraps every call in a root span that captures inputs, outputs, and timing<\/li><li><code>mlflow.update_current_trace(tags=...)<\/code>: Tags each trace with the agent and model service, so you can filter traces later<\/li><\/ul>", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom agents.tracing import set_trace_processors\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Send traces to MLflow instead of OpenAI\nset_trace_processors([])\nmlflow.set_tracking_uri(\"databricks\")\nmlflow.openai.autolog()\n\n# Read environment variables injected by the DAB\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\nmlflow.set_experiment(experiment_id=os.getenv(\"MLFLOW_EXPERIMENT_ID\"))\n\n# Define the agent\nagent = Agent(\n    name=\"Simple Chatbot\",\n    instructions=\"You are a helpful assistant. Answer the user's questions clearly and concisely.\",\n    model=MODEL_SERVICE,\n)\n\n\n@mlflow.trace(name=\"mlflow_chatbot_run\", span_type=\"AGENT\")\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent and return the final reply. The trace records each step.\"\"\"\n    mlflow.update_current_trace(tags={\"agent_name\": agent.name, \"model_service\": MODEL_SERVICE})\n    result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "databricks.yml", "desc": "<p>Replace the contents of <strong>databricks.yml<\/strong> with the version below. This adds an MLflow experiment as a bundle-managed resource and grants the app permission to write to it.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the experiment resource manually on the app Overview page during the deploy step.<\/div>", "changes": "<strong>What changed:<\/strong><ul><li><code>resources.experiments.chatbot_experiment<\/code>: Declares an MLflow experiment that DABs creates on deploy. This is a new resource added to the same app.<\/li><li><code>app-experiment<\/code> resource entry: Grants the app's service principal <code>CAN_EDIT<\/code> on the experiment so it can log traces<\/li><li>The app description updated to reflect the new capability. The bundle name and app name remain the same; redeploying updates the existing app in place.<\/li><\/ul>", "code": "bundle:\n  name: simple_chatbot\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n\ntargets:\n  dev:\n    mode: development\n    default: true"}, {"name": "app.yaml", "desc": "<p>Replace the contents of <strong>app.yaml<\/strong> with the version below. This injects the MLflow experiment ID as an environment variable.<\/p>", "changes": "<strong>What changed:<\/strong><ul><li>The new <code>MLFLOW_EXPERIMENT_ID<\/code> entry uses <code>valueFrom<\/code> to resolve the experiment ID from the <code>app-experiment<\/code> resource declared in <code>databricks.yml<\/code>. The agent reads this at startup to target the correct experiment.<\/li><\/ul>", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}], "E1": [{"name": "agent.py", "desc": "<p>Replace the contents of <strong>agent.py<\/strong> with the final version below. This adds MCP imports, a <code>WorkspaceClient<\/code> for authentication, and the URL of the Databricks-provided SQL MCP <code>system.ai.dbsql<\/code>. <code>run_agent()<\/code> now opens an <code>McpServer<\/code> for each request and passes it to the agent.<\/p>", "changes": "<strong>What changed from Section D:<\/strong><ul><li><code>McpServer<\/code> and <code>WorkspaceClient<\/code> imports added<\/li><li><code>LISTINGS_TABLE<\/code>: Your table's full name, injected by its app resource. The instructions name it, so the agent knows what to query.<\/li><li><code>async with McpServer(...)<\/code>: Connects to <code>system.ai.dbsql<\/code> through Unity Gateway for each request and closes the connection after. Its <code>execute_sql<\/code> tool runs SQL on a SQL warehouse.<\/li><li><code>mcp_servers=[mcp_server]<\/code> on the Agent: The SDK lists the MCP's tools at runtime<\/li><\/ul>", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom agents.tracing import set_trace_processors\nfrom databricks.sdk import WorkspaceClient\nfrom databricks_openai import AsyncDatabricksOpenAI\nfrom databricks_openai.agents import McpServer\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Send traces to MLflow instead of OpenAI\nset_trace_processors([])\nmlflow.set_tracking_uri(\"databricks\")\nmlflow.openai.autolog()\n\n# Read environment variables injected by the DAB\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\nmlflow.set_experiment(experiment_id=os.getenv(\"MLFLOW_EXPERIMENT_ID\"))\n\n# MCP configuration: the table's full name comes from its app resource\nLISTINGS_TABLE = os.getenv(\"LISTINGS_TABLE\")\n\nws = WorkspaceClient()\n\n# The Databricks-provided SQL MCP, reached through Unity Gateway\nDBSQL_URL = f\"{ws.config.host}/ai-gateway/mcp-services/system.ai.dbsql\"\n\nINSTRUCTIONS = (\n    \"You are a helpful assistant. Answer the user's questions clearly and concisely. \"\n    f\"For questions about San Francisco Airbnb listings, query {LISTINGS_TABLE} with the execute_sql tool.\"\n)\n\n\n@mlflow.trace(name=\"tool_calling_chatbot_run\", span_type=\"AGENT\")\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent with MCP tools and return the final reply.\"\"\"\n    mlflow.update_current_trace(tags={\"agent_name\": \"Tool-calling Chatbot\", \"model_service\": MODEL_SERVICE})\n    # Connect to the MCP for this request; leaving the block closes the connection\n    async with McpServer(url=DBSQL_URL, workspace_client=ws, name=\"dbsql\", timeout=60.0) as mcp_server:\n        # Define the agent with MCP tool access\n        agent = Agent(\n            name=\"Simple Chatbot\",\n            instructions=INSTRUCTIONS,\n            model=MODEL_SERVICE,\n            mcp_servers=[mcp_server],\n        )\n        result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "app.yaml", "desc": "<p>Replace the contents of <strong>app.yaml<\/strong> with the final version below. This adds a <code>LISTINGS_TABLE<\/code> environment variable with the full name of your listings table.<\/p>", "changes": "<strong>What changed from Section D:<\/strong><ul><li>The new entry uses <code>valueFrom<\/code> to read the table's full name from the <code>listings-table<\/code> resource, so there is nothing to type in.<\/li><\/ul>", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n  - name: LISTINGS_TABLE\n    valueFrom: listings-table\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}, {"name": "databricks.yml", "desc": "<p>Replace the contents of <strong>databricks.yml<\/strong> with the final version below. This adds a <code>variables<\/code> block for your catalog and the SQL warehouse, and two resources: <code>SELECT<\/code> on your listings table and <code>CAN_USE<\/code> on the warehouse.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the table and SQL warehouse resources manually on the app Overview page during the deploy step.<\/div>", "changes": "<strong>What changed from Section D:<\/strong><ul><li><code>variables<\/code> block: <code>catalog_name<\/code> for your catalog, and <code>warehouse_id<\/code>, which looks up the <code>shared_warehouse<\/code> SQL warehouse by name<\/li><li><code>listings-table<\/code> resource: Grants the app’s service principal <code>SELECT<\/code> on <code>sf_airbnb_listings<\/code><\/li><li><code>sql-warehouse<\/code> resource: Grants <code>CAN_USE<\/code> on the warehouse, where the SQL MCP runs the agent’s queries<\/li><li>No grant is needed for <code>system.ai.dbsql<\/code> itself: all workspace users can already call the Databricks-provided MCPs<\/li><\/ul>", "code": "bundle:\n  name: simple_chatbot\n\nvariables:\n  catalog_name:\n    description: \"Per-user lab catalog, e.g. labuser_matthew_mccoy\"\n    default: # <---- Update with your catalog name\n  warehouse_id:\n    description: \"SQL warehouse the SQL MCP runs queries on\"\n    lookup:\n      warehouse: \"shared_warehouse\"\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n        - name: listings-table\n          uc_securable:\n            securable_full_name: ${var.catalog_name}.agent_deployment.sf_airbnb_listings\n            securable_type: TABLE\n            permission: SELECT\n        - name: sql-warehouse\n          sql_warehouse:\n            id: ${var.warehouse_id}\n            permission: CAN_USE\n\ntargets:\n  dev:\n    mode: development\n    default: true"}]}, STAGE = 1, P = "d1";
  var ORDER = ["agent.py", "server.py", "pyproject.toml", "app.yaml", "databricks.yml", "requirements.txt"];
  var LANG = { py: "python", toml: "toml", yaml: "yaml", yml: "yaml", txt: "text" };
  var PYPROJECT_DESC = '<p>Add <code>"mlflow==3.16.1"</code> to the <code>dependencies</code> list in <strong>pyproject.toml</strong>. Without this dependency, the app cannot import <code>mlflow</code> at runtime and will crash on startup.</p>';
  function buildStages() {
    function toMap(list, key) { var m = {}; list.forEach(function (f) { m[f.name] = f[key]; }); return m; }
    var s1 = toMap(RAW.C1, "code"), s2 = Object.assign({}, s1, toMap(RAW.D1, "code"));
    s2["pyproject.toml"] = s1["pyproject.toml"].replace('    "uv",', '    "uv",\n    "mlflow==3.16.1",');
    var s3 = Object.assign({}, s2, toMap(RAW.E1, "code"));
    var metas = [RAW.C1, RAW.D1, RAW.E1].map(function (list) { var m = {}; list.forEach(function (f) { m[f.name] = { desc: f.desc || "", changes: f.changes || "" }; }); return m; });
    metas[1]["pyproject.toml"] = { desc: PYPROJECT_DESC, changes: "" };
    var stages = [s1, s2, s3].map(function (files, i) { return { files: files, meta: metas[i] }; });
    stages.forEach(function (st, i) {
      var prev = i ? stages[i - 1].files : {};
      st.prev = prev;
      st.changed = ORDER.filter(function (n) { return prev[n] !== st.files[n]; });
    });
    return stages;
  }
  function langOf(name) { return LANG[name.split(".").pop()] || "text"; }
  function esc(s) { return s.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;"); }
  function hlLine(line, lang) {
    var re;
    if (lang === "python") re = /(#.*$)|("""[\s\S]*?"""|"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:from|import|def|class|return|if|elif|else|for|with|as|async|await|None|True|False|and|or|not|in)\b)|(@[\w.]+)|(\b\w+(?=\())|(\b\d+\.?\d*\b)/g;
    else if (lang === "yaml" || lang === "toml") re = /(#.*$)|("(?:[^"\\]|\\.)*")|(\$\{[^}]+\})|(^\s*-?\s*)([\w][\w\-.]*)(?=\s*[:=])|(\[[\w.\-]+\])/g;
    else return esc(line);
    var out = "", last = 0, m;
    while ((m = re.exec(line))) {
      out += esc(line.slice(last, m.index));
      if (lang === "python") {
        var cls = m[1] ? "c" : m[2] ? "s" : m[3] ? "k" : m[4] ? "d" : m[5] ? "f" : "n";
        out += '<span class="t-' + cls + '">' + esc(m[0]) + "</span>";
      } else if (m[1]) out += '<span class="t-c">' + esc(m[0]) + "</span>";
      else if (m[2]) out += '<span class="t-s">' + esc(m[0]) + "</span>";
      else if (m[3]) out += '<span class="t-d">' + esc(m[0]) + "</span>";
      else if (m[5]) out += esc(m[4]) + '<span class="t-y">' + esc(m[5]) + "</span>";
      else out += '<span class="t-k">' + esc(m[0]) + "</span>";
      last = m.index + m[0].length;
      if (m[0] === "") re.lastIndex++;
    }
    return out + esc(line.slice(last));
  }
  function addedSet(a, b) {
    a = a ? a.split("\n") : []; b = b.split("\n");
    var n = a.length, m = b.length, dp = [], i, j, set = {}, k = 0;
    for (i = 0; i <= n; i++) dp.push(new Array(m + 1).fill(0));
    for (i = n - 1; i >= 0; i--) for (j = m - 1; j >= 0; j--) dp[i][j] = a[i] === b[j] ? dp[i + 1][j + 1] + 1 : Math.max(dp[i + 1][j], dp[i][j + 1]);
    i = 0; j = 0;
    while (j < m) {
      if (i < n && a[i] === b[j]) { i++; j++; }
      else if (i < n && dp[i + 1][j] >= dp[i][j + 1]) i++;
      else set[j++] = 1;
    }
    return set;
  }
  var COPY_ICON = '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>';
  function copyText(text, btn) {
    var t = document.createElement("textarea"); t.value = text; t.style.position = "fixed"; t.style.opacity = "0"; document.body.appendChild(t); t.select();
    var ok = false; try { ok = document.execCommand("copy"); } catch (e) {} document.body.removeChild(t);
    if (!ok && navigator.clipboard) { navigator.clipboard.writeText(text); ok = true; }
    if (ok) { btn.classList.add("ok"); btn.innerHTML = "Copied"; setTimeout(function () { btn.classList.remove("ok"); btn.innerHTML = COPY_ICON + "Copy"; }, 1500); }
  }
  var S = buildStages(), st = S[STAGE];
  var CONTS = [
    { id: "app", x: 215, y: 84, w: 505, h: 296, label: "DATABRICKS APP" },
    { id: "gw", x: 790, y: 40, w: 270, h: STAGE < 2 ? 180 : 340, label: "UNITY GATEWAY" }
  ];
  var NODES = [
    { id: "browser", x: 14, y: 170, w: 122, h: 84, t: "Browser", s: "chat.html", from: 0 },
    { id: "server", x: 240, y: 170, w: 175, h: 84, t: "server.py", s: "FastAPI", from: 0, mono: 1 },
    { id: "agent", x: 522, y: 170, w: 175, h: 84, t: "agent.py", s: "OpenAI Agents SDK", from: 0, mono: 1 },
    { id: "runtime", x: 240, y: 292, w: 175, h: 62, t: "uv run start-server", s: "", from: 0, mono: 1, small: 1 },
    { id: "model", x: 815, y: 104, w: 220, h: 84, t: "agent-model", s: "Model service", from: 0, perm: "EXECUTE", mono: 1 },
    { id: "mcp", x: 815, y: 264, w: 220, h: 84, t: "system.ai.dbsql", s: "MCP server", from: 2, mono: 1 },
    { id: "warehouse", x: 1100, y: 264, w: 200, h: 84, t: "shared_warehouse", s: "SQL warehouse", from: 2, perm: "CAN_USE", mono: 1 },
    { id: "table", x: 1100, y: 416, w: 200, h: 84, t: "sf_airbnb_listings", s: "Unity Catalog table", from: 2, perm: "SELECT", mono: 1 },
    { id: "exp", x: 479, y: 420, w: 262, h: 84, t: "MLflow experiment", s: ["", "Traces", "Traces + tool spans"][STAGE], from: 1, perm: "CAN_EDIT" }
  ];
  var EDGES = [
    { a: "browser", b: "server", l: "/invocations", from: 0 },
    { a: "server", b: "agent", l: "run_agent()", from: 0 },
    { a: "agent", b: "model", l: "Responses API", from: 0, end: 1 },
    { a: "agent", b: "mcp", l: "execute_sql", from: 2, end: 1 },
    { a: "mcp", b: "warehouse", l: "SQL", from: 2 },
    { a: "warehouse", b: "table", l: "", from: 2, down: 1 },
    { a: "agent", b: "exp", l: "", from: 1, down: 1 }
  ];
  // [file, line prefix, include the indented block that follows]
  var MAP = {
    browser: [["server.py", "CHAT_UI"], ["server.py", "@app.get"], ["server.py", "async def root"], ["server.py", "return CHAT_UI"]],
    server: [["server.py", "import uvicorn"], ["server.py", "from fastapi"], ["server.py", "from pydantic"], ["server.py", "from agent import"], ["server.py", "app = FastAPI"],
      ["server.py", "class ChatRequest", 1], ["server.py", "@app.post"], ["server.py", "async def invocations", 1], ["server.py", "def main", 1]],
    agent: [["agent.py", "from agents import"], ["agent.py", "# Define"], ["agent.py", "agent = Agent(", 1], ["agent.py", "async def run_agent"], ["agent.py", "result = await Runner.run"], ["agent.py", "return"], ["agent.py", "final_output = result"]],
    runtime: [["app.yaml", "command:"], ["pyproject.toml", "dependencies = [", 1], ["pyproject.toml", "[tool.hatch"], ["pyproject.toml", "include ="], ["pyproject.toml", "[project.scripts]"], ["pyproject.toml", "start-server"], ["requirements.txt", "uv"]],
    app: [["databricks.yml", "bundle:", 1], ["databricks.yml", "apps:"], ["databricks.yml", "simple_chatbot:"], ["databricks.yml", "name: agent-"], ["databricks.yml", "description:"], ["databricks.yml", "source_code_path:"], ["databricks.yml", "resources:"], ["databricks.yml", "targets:", 1]],
    gw: [["agent.py", "set_default_openai_client"], ["agent.py", "DBSQL_URL ="], ["agent.py", "# The Databricks-provided"]],
    model: [["agent.py", "from databricks_openai import AsyncDatabricksOpenAI"], ["agent.py", "# Point the"], ["agent.py", "set_default_openai_client"], ["agent.py", "set_default_openai_api"], ["agent.py", "# Read the model"],
      ["agent.py", "MODEL_SERVICE ="], ["agent.py", "model=MODEL_SERVICE"], ["app.yaml", "- name: MODEL_SERVICE", 1], ["databricks.yml", "- name: model-service", 1]],
    exp: [["agent.py", "from agents.tracing"], ["agent.py", "import mlflow"], ["agent.py", "# Send traces"], ["agent.py", "set_trace_processors("], ["agent.py", "mlflow.set_tracking_uri"], ["agent.py", "mlflow.openai.autolog"], ["agent.py", "mlflow.set_experiment"],
      ["agent.py", "@mlflow.trace"], ["agent.py", "mlflow.update_current_trace"], ["app.yaml", "- name: MLFLOW_EXPERIMENT_ID", 1], ["databricks.yml", "experiments:", 1], ["databricks.yml", "- name: app-experiment", 1], ["pyproject.toml", "\"mlflow=="]],
    mcp: [["agent.py", "from databricks_openai.agents"], ["agent.py", "from databricks.sdk"], ["agent.py", "ws = WorkspaceClient"], ["agent.py", "# The Databricks-provided"], ["agent.py", "DBSQL_URL ="], ["agent.py", "# Connect to the MCP"], ["agent.py", "async with McpServer("], ["agent.py", "mcp_servers="]],
    warehouse: [["databricks.yml", "warehouse_id:", 1], ["databricks.yml", "- name: sql-warehouse", 1]],
    table: [["agent.py", "# MCP configuration"], ["agent.py", "LISTINGS_TABLE ="], ["agent.py", "f\"For questions"], ["app.yaml", "- name: LISTINGS_TABLE", 1], ["databricks.yml", "catalog_name:", 1], ["databricks.yml", "- name: listings-table", 1]]
  };
  var sel = null, file = st.changed[0];
  function $(id) { return document.getElementById(P + "-" + id); }
  function ind(l) { return l.length - l.replace(/^\s+/, "").length; }
  function hits(id, name) {
    var lines = st.files[name].split("\n"), set = {};
    (MAP[id] || []).forEach(function (m) {
      if (m[0] !== name) return;
      lines.forEach(function (l, i) {
        if (l.trim().indexOf(m[1]) !== 0) return;
        set[i] = 1;
        if (!m[2]) return;
        var base = ind(l), j = i + 1;
        while (j < lines.length && (!lines[j].trim() || ind(lines[j]) > base)) { if (lines[j].trim()) set[j] = 1; j++; }
        if (j < lines.length && /^[\)\]]/.test(lines[j].trim()) && ind(lines[j]) === base) set[j] = 1;
      });
    });
    return set;
  }
  function count(o) { return Object.keys(o).length; }
  function N(id) { return NODES.filter(function (n) { return n.id === id; })[0]; }
  function vis(o) { return o.from <= STAGE; }
  function isNew(o) { return STAGE > 0 && o.from === STAGE; }
  function draw() {
    var h = '<defs><marker id="' + P + '-ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#9AABB1"/></marker>' +
      '<marker id="' + P + '-arn" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#FF5F46"/></marker></defs>';
    CONTS.forEach(function (c) {
      h += '<g class="cont' + (sel === c.id ? " sel" : "") + '"><rect class="box" x="' + c.x + '" y="' + c.y + '" width="' + c.w + '" height="' + c.h + '" rx="16"/>' +
        '<g class="hdr" data-id="' + c.id + '"><rect x="' + c.x + '" y="' + c.y + '" width="' + c.w + '" height="40" rx="16" fill="transparent"/><text x="' + (c.x + 22) + '" y="' + (c.y + 27) + '">' + c.label + "</text></g></g>";
    });
    EDGES.forEach(function (e) {
      if (!vis(e)) return;
      var a = N(e.a), b = N(e.b), d, lx, x1, y1, x2, y2;
      if (e.down) { x1 = a.x + a.w / 2; y1 = a.y + a.h; d = "M" + x1 + "," + y1 + " L" + x1 + "," + (b.y - 2); }
      else {
        x1 = a.x + a.w; y1 = a.y + a.h / 2; x2 = b.x; y2 = b.y + b.h / 2; lx = (x1 + x2) / 2;
        d = "M" + x1 + "," + y1 + " C" + lx + "," + y1 + " " + lx + "," + y2 + " " + (x2 - 2) + "," + y2;
      }
      var lab = !e.l ? "" : e.end ? '<text x="' + (b.x + 4) + '" y="' + (b.y - 9) + '">' + e.l + "</text>" : '<text x="' + lx + '" y="' + (y1 - 10) + '" text-anchor="middle">' + e.l + "</text>";
      h += '<g class="ed' + (isNew(e) ? " new" : "") + '"><path d="' + d + '" marker-end="url(#' + P + (isNew(e) ? "-arn" : "-ar") + ')"/>' + lab + "</g>";
    });
    NODES.forEach(function (n) {
      if (!vis(n)) return;
      var cy = n.y + n.h / 2;
      h += '<g class="nd' + (n.mono ? " mono" : "") + (n.small ? " small" : "") + (isNew(n) ? " new" : "") + (sel === n.id ? " sel" : "") + '" data-id="' + n.id + '">' +
        '<rect class="box" x="' + n.x + '" y="' + n.y + '" width="' + n.w + '" height="' + n.h + '" rx="10"/>' +
        '<text class="t" x="' + (n.x + n.w / 2) + '" y="' + (n.s ? cy - 4 : cy + 5) + '" text-anchor="middle">' + n.t + "</text>" +
        (n.s ? '<text class="s" x="' + (n.x + n.w / 2) + '" y="' + (cy + 17) + '" text-anchor="middle">' + n.s + "</text>" : "");
      if (n.perm) { var pw = n.perm.length * 7.6 + 16; h += '<g class="pill"><rect x="' + (n.x + n.w - pw - 10) + '" y="' + (n.y - 10) + '" width="' + pw + '" height="20" rx="10"/><text x="' + (n.x + n.w - 10 - pw / 2) + '" y="' + (n.y + 4) + '" text-anchor="middle">' + n.perm + "</text></g>"; }
      h += "</g>";
    });
    var svg = $("svg");
    svg.setAttribute("viewBox", "0 30 " + (STAGE < 2 ? 1080 : 1320) + " " + (STAGE < 1 ? 360 : 486));
    svg.innerHTML = h;
    Array.prototype.forEach.call(svg.querySelectorAll("[data-id]"), function (g) { g.addEventListener("click", function () { pick(g.getAttribute("data-id")); }); });
  }
  function pick(id) {
    sel = sel === id ? null : id;
    if (sel && !count(hits(sel, file))) {
      var withHits = ORDER.filter(function (n) { return count(hits(sel, n)); });
      file = withHits.filter(function (n) { return st.changed.indexOf(n) >= 0; })[0] || withHits[0] || file;
    }
    render();
  }
  function render() {
    draw();
    var t = $("tabs");
    t.innerHTML = ORDER.map(function (n) {
      var ch = st.changed.indexOf(n) >= 0, k = sel ? count(hits(sel, n)) : 0;
      return '<div class="tab' + (n === file ? " on" : "") + (ch ? "" : " same") + '" data-f="' + n + '">' + n + (STAGE && ch ? '<span class="d"></span>' : "") + (k ? '<span class="n">' + k + "</span>" : "") + "</div>";
    }).join("") + '<span class="grow"></span><button class="cp" id="' + P + '-cp">' + COPY_ICON + "Copy</button>";
    Array.prototype.forEach.call(t.querySelectorAll(".tab"), function (el) { el.addEventListener("click", function () { file = el.getAttribute("data-f"); render(); }); });
    var meta = st.changed.indexOf(file) >= 0 && st.meta[file];
    $("desc").innerHTML = meta ? '<div class="desc">' + meta.desc + (meta.changes ? '<div class="chg">' + meta.changes + "</div>" : "") + "</div>" : "";
    $("split").className = "split" + (meta ? "" : " full");
    var hs = sel ? hits(sel, file) : null, add = STAGE && st.changed.indexOf(file) >= 0 ? addedSet(st.prev[file], st.files[file]) : {};
    var c = $("code");
    c.className = "code" + (hs && count(hs) ? " focus" : "");
    c.innerHTML = st.files[file].split("\n").map(function (l, i) {
      return '<div class="ln' + (add[i] ? " add" : "") + (hs && hs[i] ? " hit" : "") + '"><span class="g">' + (add[i] ? "+" : i + 1) + "</span><span>" + (hlLine(l, langOf(file)) || " ") + "</span></div>";
    }).join("");
    c.style.maxHeight = ""; /* height now follows the description panel via CSS grid stretch */
    var first = c.querySelector(".hit"); if (first) c.scrollTop = first.offsetTop - c.offsetTop - 40;
    var b = $("cp"); b.onclick = function () { copyText(st.files[file], b); };
  }
  render();
})();
</script>

In [0]:
uniqueIdentifier2 = user_email.split("@")[0].replace('labuser','labuser_').replace(".","_")
print(f"[dev {uniqueIdentifier2}] simple-chatbot-experiment")

### D2. Redeploy

Redeploy using whichever method you used in Section C2. This update adds an MLflow experiment resource to the app. Select a card below.

<div style="box-sizing: border-box; margin: 16px 4px; font-family: sans-serif; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 12px; padding: 16px 22px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
<div style="height: 4px; border-radius: 2px; margin: -2px 0 14px; background: #1B5162;"></div>
<div style="display:flex; align-items:center; justify-content:space-between; gap:12px; flex-wrap:wrap;">
  <strong style="font-size:15pt; color:#0b2026;">Grant the app access to the MLflow trace tables</strong>
  <span style="font-family:Menlo,Consolas,monospace; font-size:10.5pt; font-weight:700; letter-spacing:.04em; color:#1B5162; border:1.5px solid #1B5162; border-radius:999px; padding:2px 10px;">SELECT · MODIFY</span>
</div>
<p style="font-size:14pt; line-height:1.6; color:#333; margin:10px 0 14px 0;">When your experiment stores traces in <strong>Unity Catalog</strong>, MLflow writes them to four tables in <code>&amp;lt;catalog&amp;gt;.agent_deployment</code>. The app's service principal needs <strong>Can select and modify</strong> on all four, or the app cannot write traces.</p>
<div style="display:grid; grid-template-columns:1fr 1fr; gap:10px;">
  <div style="background:#F9F7F4; border:1px solid #EEEDE9; border-left:5px solid #1B5162; border-radius:8px; padding:12px 14px;"><div style="display:flex; align-items:center; gap:10px;"><svg width="20" height="20" viewBox="0 0 16 16" style="flex:none;"><path fill-rule="evenodd" clip-rule="evenodd" d="M1 1.75C1 1.33579 1.33579 1 1.75 1H14.25C14.6642 1 15 1.33579 15 1.75V14.25C15 14.6642 14.6642 15 14.25 15H1.75C1.33579 15 1 14.6642 1 14.25V1.75ZM2.5 2.5V5.5H13.5V2.5H2.5ZM2.5 13.5V7H5V13.5H2.5ZM6.5 13.5H9.5V7H6.5V13.5ZM11 7V13.5H13.5V7H11Z" fill="#0B2026"/></svg><code style="font-size:13.5pt; font-weight:700; background:none; color:#0b2026;"><span style="color:#FF5F46;">&amp;lt;prefix&amp;gt;</span>_otel_spans</code></div><div style="margin-top:6px; font-size:12.5pt; color:#618794; line-height:1.45;">One row per span: the call tree with timing, inputs and outputs</div></div>
  <div style="background:#F9F7F4; border:1px solid #EEEDE9; border-left:5px solid #1B5162; border-radius:8px; padding:12px 14px;"><div style="display:flex; align-items:center; gap:10px;"><svg width="20" height="20" viewBox="0 0 16 16" style="flex:none;"><path fill-rule="evenodd" clip-rule="evenodd" d="M1 1.75C1 1.33579 1.33579 1 1.75 1H14.25C14.6642 1 15 1.33579 15 1.75V14.25C15 14.6642 14.6642 15 14.25 15H1.75C1.33579 15 1 14.6642 1 14.25V1.75ZM2.5 2.5V5.5H13.5V2.5H2.5ZM2.5 13.5V7H5V13.5H2.5ZM6.5 13.5H9.5V7H6.5V13.5ZM11 7V13.5H13.5V7H11Z" fill="#0B2026"/></svg><code style="font-size:13.5pt; font-weight:700; background:none; color:#0b2026;"><span style="color:#FF5F46;">&amp;lt;prefix&amp;gt;</span>_otel_annotations</code></div><div style="margin-top:6px; font-size:12.5pt; color:#618794; line-height:1.45;">Assessments, feedback and tags attached to traces</div></div>
  <div style="background:#F9F7F4; border:1px solid #EEEDE9; border-left:5px solid #1B5162; border-radius:8px; padding:12px 14px;"><div style="display:flex; align-items:center; gap:10px;"><svg width="20" height="20" viewBox="0 0 16 16" style="flex:none;"><path fill-rule="evenodd" clip-rule="evenodd" d="M1 1.75C1 1.33579 1.33579 1 1.75 1H14.25C14.6642 1 15 1.33579 15 1.75V14.25C15 14.6642 14.6642 15 14.25 15H1.75C1.33579 15 1 14.6642 1 14.25V1.75ZM2.5 2.5V5.5H13.5V2.5H2.5ZM2.5 13.5V7H5V13.5H2.5ZM6.5 13.5H9.5V7H6.5V13.5ZM11 7V13.5H13.5V7H11Z" fill="#0B2026"/></svg><code style="font-size:13.5pt; font-weight:700; background:none; color:#0b2026;"><span style="color:#FF5F46;">&amp;lt;prefix&amp;gt;</span>_otel_logs</code></div><div style="margin-top:6px; font-size:12.5pt; color:#618794; line-height:1.45;">OpenTelemetry log records emitted while the app runs</div></div>
  <div style="background:#F9F7F4; border:1px solid #EEEDE9; border-left:5px solid #1B5162; border-radius:8px; padding:12px 14px;"><div style="display:flex; align-items:center; gap:10px;"><svg width="20" height="20" viewBox="0 0 16 16" style="flex:none;"><path fill-rule="evenodd" clip-rule="evenodd" d="M1 1.75C1 1.33579 1.33579 1 1.75 1H14.25C14.6642 1 15 1.33579 15 1.75V14.25C15 14.6642 14.6642 15 14.25 15H1.75C1.33579 15 1 14.6642 1 14.25V1.75ZM2.5 2.5V5.5H13.5V2.5H2.5ZM2.5 13.5V7H5V13.5H2.5ZM6.5 13.5H9.5V7H6.5V13.5ZM11 7V13.5H13.5V7H11Z" fill="#0B2026"/></svg><code style="font-size:13.5pt; font-weight:700; background:none; color:#0b2026;"><span style="color:#FF5F46;">&amp;lt;prefix&amp;gt;</span>_otel_metrics</code></div><div style="margin-top:6px; font-size:12.5pt; color:#618794; line-height:1.45;">OpenTelemetry metric data points recorded alongside traces</div></div>
</div>
<p style="font-size:12.5pt; color:#618794; margin:12px 0 0 0;"><code>&amp;lt;prefix&amp;gt;</code> defaults to the experiment ID. The steps are in the <strong>Apps UI</strong> card below.</p>
</div>

<br></br>

<div style="width: 100%; font-family: sans-serif;">

<style>
.d2d-row { padding: 4px 6px; box-sizing: border-box; display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
.d2d-box {
  display: block; background: #fff; border: 2px solid #C6DCEA; box-sizing: border-box; border-radius: 12px;
  padding: 16px 20px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);
  cursor: pointer; user-select: none; text-align: left;
  transition: transform 0.12s, box-shadow 0.15s, border-color 0.15s;
}
.d2d-box::before { content: ""; display: block; height: 4px; border-radius: 2px; background: var(--pc); margin: -2px 0 14px; }
.d2d-box:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(27,49,57,.10); }
.d2d-box.active { border-color: var(--pc); box-shadow: 0 6px 16px rgba(27,49,57,.10); }
.d2d-pill {
  display: inline-block; background: var(--pc); color: #fff; font-size: 10.5pt; font-weight: 700;
  letter-spacing: 0.06em; border-radius: 999px; padding: 3px 12px; pointer-events: none;
}
.d2d-title { display: block; font-size: 16pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 10px; pointer-events: none; }
.d2d-subtitle { display: block; font-size: 12.5pt; font-weight: 500; color: #5A6F77; line-height: 1.3; margin-top: 3px; pointer-events: none; }
.d2d-detail-wrap {
  padding: 0 6px 8px; box-sizing: border-box; overflow: hidden; max-height: 0; opacity: 0; margin-top: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
}
.d2d-detail-wrap.open { max-height: 2400px; opacity: 1; margin-top: 16px; }
.d2d-detail-card {
  background: #fff; border: 1.5px solid #C6DCEA; border-radius: 12px;
  padding: 18px 22px 20px; box-sizing: border-box; box-shadow: 0 2px 8px rgba(27,49,57,.07);
}
.d2d-accent { height: 4px; border-radius: 2px; margin: -2px 0 14px; }
.d2d-steps-block { font-size: 14.5pt; line-height: 1.6; color: #0b2026; margin-bottom: 14px; }
.d2d-steps-block ol { margin: 8px 0 0 0; padding-left: 22px; }
.d2d-steps-block li { margin-bottom: 8px; }
.d2d-steps-block ul { margin: 4px 0 8px 20px; list-style: disc; }
.d2d-info-block {
  background: #F9F7F4; border-left: 4px solid; border-radius: 0 6px 6px 0;
  padding: 12px 14px; font-size: 14pt; line-height: 1.5; color: #0b2026;
}
</style>

<div>
  <div class="d2d-row">
    <span class="d2d-box active" data-id="0" onclick="d2dSelect(0)" style="--pc:#5A6F77;">
      <span class="d2d-pill">DABS UI</span>
      <span class="d2d-title">Git folder</span>
      <span class="d2d-subtitle">Deploys from databricks.yml</span>
    </span>
    <span class="d2d-box" data-id="1" onclick="d2dSelect(1)" style="--pc:#FF5F46;">
      <span class="d2d-pill">APPS UI</span>
      <span class="d2d-title">No Git folder</span>
      <span class="d2d-subtitle">Add resources by hand</span>
    </span>
  </div>

  <div class="d2d-detail-wrap" id="d2d-detail-wrap">
    <div class="d2d-detail-card" id="d2d-detail-card"></div>
  </div>

</div>

</div>

<script>
var D2D_CARDS = [
  {
    title: "DABs UI (Git folder)",
    color: "#5A6F77",
    bg: "rgba(90,111,119,0.10)",
    infoBg: "rgba(90,111,119,0.10)",
    content: `
      <ol>
        <li>Click the rocket icon on any file in the project folder.</li>
        <li>Click <strong>Deploy</strong>, wait for validation, then click <strong>Deploy</strong> again.</li>
        <li>Navigate to the app and click <strong>Deploy</strong> to push the updated source.</li>
        <li>Notice that you now have a new <strong>App resource</strong> for your <strong>MLflow experiment</strong>.</li>
        <li>After deployment, click on the URL to open the app.</li>
      </ol>
    `,
    info: '<strong>Requires a Git folder.</strong> The DABs UI detects the new experiment resource in <code>databricks.yml</code> and creates it automatically. Wait for the app status to return to <strong>Running</strong>.'
  },
  {
    title: "Apps UI (no Git folder)",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    infoBg: "rgba(255,95,70,0.10)",
    content: `
      <ol>
        <li>Navigate to the app Overview page and click <strong>(⋮)</strong> icon then on  <strong>Edit</strong> at the top right.</li>
        <li>Click <strong>resources</strong> at the top of the setup wizard.</li>
        <li>Click <strong>+ Add resource</strong> and select <strong>MLflow experiment</strong>.</li>
        <li>In the <strong>Select experiment</strong> dropdown, select <strong>Create new experiment</strong>.</li>
        <li>Give it the name <strong>[dev labuser_XXX] simple-chatbot-experiment</strong>, using your own labuser name as printed above.</li>
        <li>For the <strong>trace storage location</strong>, select <strong>Unity Catalog</strong>. The other option, <strong>In the Experiment</strong>, stores traces in MLflow's managed storage. Databricks recommends Unity Catalog for production workloads. Select the <code>agent_deployment</code> schema in your user catalog and leave the prefix as the default value.</li> 
        <li>Click <strong>Create experiment</strong>.</li>
        <li>Make sure <strong>Permission</strong> is set to <strong>Can edit</strong> and <strong>Resource key</strong> is set to <strong>app-experiment</strong>.</li>
        <li>Click <strong>Save</strong>.</li>
        <li>Next, grant the app permissions to the underlying MLflow tables so it can write to them. Navigate back to your app's <strong>Overview</strong> page, click the <strong>(⋮)</strong> icon, then click <strong>Edit</strong> at the top right.</li>
        <li>Under <strong>Resources</strong>, click <strong>+ Add resource</strong> and select <strong>UC table</strong>.</li>
        <li>Click the folder icon under <strong>UC table</strong> and search for <strong>otel_annotations</strong>. Click the table name, then click <strong>Confirm</strong>.</li>
        <li>Set <strong>Permission</strong> to <strong>Can select and modify</strong> so the app can read and write to the underlying trace objects in <strong>Experiments</strong>.</li>
        <li>As this is the first table, update the <strong>Resource key</strong> for <strong>otel_annotations</strong> to <strong>table-1</strong>.</li>
        <li>Repeat the process to add the other 3 MLflow tables with the same permissions, then click <strong>Save</strong>.</li>
        <li>Back on the Overview page, click <strong>Deploy</strong> to push the updated source.</li>
      </ol>
    `,
    info: '<strong>No Git folder required.</strong> You add the experiment resource manually on the Overview page. Wait for the app status to return to <strong>Running</strong>.'
  }
];

var d2dCurrent = null;

function d2dSelect(id) {
  var wrap = document.getElementById('d2d-detail-wrap');
  var card = document.getElementById('d2d-detail-card');
  var c = D2D_CARDS[id];

  document.querySelectorAll('.d2d-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (d2dCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.d2d-box').forEach(function(b) {
      b.classList.remove('active');
    });
    d2dCurrent = null;
    return;
  }

  d2dCurrent = id;

  var label = c.title.split(' (')[0].toUpperCase();
  var sub = c.title.replace(/^.*\((.*)\)$/, '$1'); sub = sub.charAt(0).toUpperCase() + sub.slice(1);
  card.innerHTML =
    '<div class="d2d-accent" style="background:' + c.color + ';"></div>'
    + '<span class="d2d-pill" style="--pc:' + c.color + ';">' + label + '</span>'
    + '<div style="font-size:18pt;font-weight:700;margin:10px 0 12px;color:#0b2026;">' + sub + '</div>'
    + '<div class="d2d-steps-block">'
    + c.content
    + '</div>'
    + '<div class="d2d-info-block" style="border-color:' + c.color + ';">'
    + c.info
    + '</div>';

  wrap.classList.add('open');
}

d2dSelect(0);
</script>

### D3. Test and Verify Traces

With MLflow tracing wired in, every conversation now produces a structured trace. Verify this is working before moving on.

1. Open the app URL and send the same query as last time: "Please tell me about Apache Spark."
2. Wait for the agent to respond.
3. Click **Experiments** in the left workspace sidebar, then search for and open the experiment whose name is printed by the code cell before D2: `dev your_username simple-chatbot-experiment`.
4. Click the **Traces** tab. You should see a trace entry for the query you just sent.
5. Click the trace to expand it. You will see:
   - A **root span** (`mlflow_chatbot_run` is located under **Details & Timeline**) capturing the full agent invocation with inputs, output, and total latency
   - **Child spans** that `mlflow.openai.autolog()` records for each Agents SDK step: the agent (`Simple Chatbot`) and its model call (`Response`)
   - The **tags** `agent_name` and `model_service`, added by `mlflow.update_current_trace()`

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">At this stage, the trace shows the LLM interaction but no tool calls. The agent can only chat. In the next section, you will add MCP tools and see tool call spans appear as children within the trace.</p>
</div>
</div>
</div>

## E. Add MCP Tools

With tracing in place, you will now extend the agent to call external tools via the Model Context Protocol (MCP). MCP is an open standard that lets AI applications discover and invoke tools hosted on external servers. Here you connect the agent to <code>system.ai.dbsql</code>, a Databricks-provided MCP whose <code>execute_sql</code> tool runs SQL on a SQL warehouse, so the agent can answer questions from your listings table.

### E1. Final Files

MCP tools add the SQL MCP, a SQL warehouse and your listings table, and change three files. Green lines are new. Select a component in the diagram to highlight its code.

<style>
#e1-arch { width:100%; margin:12px 0 0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; color:#0b2026; }
#e1-arch .dia { margin:0 8px; background:#fff; border:1px solid #DCE0E2; border-radius:14px; padding:12px; box-shadow:0 4px 18px rgba(27,49,57,.06); }
#e1-arch svg { display:block; width:100%; height:auto; }
#e1-arch svg text { font-family:inherit; fill:#0b2026; }
#e1-arch .cont rect.box { fill:#F9F7F4; stroke:#DCE0E2; stroke-width:1.5; }
#e1-arch .cont .hdr { cursor:pointer; }
#e1-arch .cont .hdr text { font-size:13px; font-weight:700; letter-spacing:1.4px; fill:#618794; }
#e1-arch .cont .hdr:hover text, #e1-arch .cont.sel .hdr text { fill:#FF5F46; }
#e1-arch .nd { cursor:pointer; }
#e1-arch .nd rect.box { fill:#fff; stroke:#C9D0D3; stroke-width:1.5; transition:fill .15s, stroke .15s; }
#e1-arch .nd:hover rect.box { stroke:#1B5162; }
#e1-arch .nd .t { font-size:16px; font-weight:700; }
#e1-arch .nd .s { font-size:13px; fill:#618794; }
#e1-arch .nd.mono .t { font-family:ui-monospace,Menlo,Consolas,monospace; font-size:15px; }
#e1-arch .nd.small .t { font-size:13.5px; }
#e1-arch .nd.new rect.box { stroke:#FF5F46; stroke-width:2.2; }
#e1-arch .nd.sel rect.box { fill:#1B5162; stroke:#1B5162; }
#e1-arch .nd.sel .t, #e1-arch .nd.sel .s { fill:#fff; }
#e1-arch .pill rect { fill:#fff; stroke:#1B5162; stroke-width:1.2; }
#e1-arch .pill text { font-size:10.5px; font-weight:700; letter-spacing:.6px; fill:#1B5162; font-family:ui-monospace,Menlo,Consolas,monospace; }
#e1-arch .ed path { fill:none; stroke:#9AABB1; stroke-width:2; }
#e1-arch .ed.new path { stroke:#FF5F46; }
#e1-arch .ed text { font-size:12px; fill:#618794; font-family:ui-monospace,Menlo,Consolas,monospace; }
#e1-arch .ed.new text { fill:#FF5F46; }
#e1-arch .panel { margin-top:14px; border:1px solid #DCE0E2; border-radius:12px; overflow:hidden; background:#fff; }
#e1-arch .tabs { display:flex; flex-wrap:wrap; align-items:flex-end; gap:2px; background:#F9F7F4; border-bottom:1px solid #DCE0E2; padding:6px 8px 0; }
#e1-arch .tab { padding:8px 13px; font-size:13px; font-weight:600; color:#618794; border-radius:7px 7px 0 0; cursor:pointer; display:flex; gap:6px; align-items:center; font-family:ui-monospace,Menlo,Consolas,monospace; }
#e1-arch .tab.same { color:#AEB8BC; }
#e1-arch .tab.on { background:#fff; color:#0b2026; box-shadow:0 -1px 0 #DCE0E2, 1px 0 0 #DCE0E2, -1px 0 0 #DCE0E2; }
#e1-arch .tab .d { width:7px; height:7px; border-radius:50%; background:#FF5F46; }
#e1-arch .tab .n { font-size:10.5px; font-weight:700; color:#fff; background:#1B5162; border-radius:999px; padding:1px 6px; }
#e1-arch .grow { flex:1; }
#e1-arch .cp { display:inline-flex; flex:none; white-space:nowrap; min-width:84px; justify-content:center; align-items:center; gap:6px; margin-bottom:6px; border:1px solid #1B3139; background:#1B3139; color:#fff; padding:6px 12px; border-radius:6px; font-size:13px; font-weight:600; cursor:pointer; }
#e1-arch .cp.ok { background:#00A972; border-color:#00A972; }
#e1-arch .split { display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; align-items:start; }
#e1-arch .split.full { grid-template-columns:1fr; }
#e1-arch .split.full .side { display:none; }
#e1-arch .side { border-right:1px solid #EEEDE9; background:#fff; min-width:0; overflow-wrap:anywhere; }
#e1-arch .side code { overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
@media (min-width: 821px) { #e1-arch .split:not(.full) { align-items:stretch; grid-template-rows:minmax(420px, auto); } #e1-arch .split:not(.full) .code { height:0; min-height:100%; max-height:none; } }
@media (max-width: 820px) { #e1-arch .split { grid-template-columns:1fr; } #e1-arch .side { border-right:none; border-bottom:1px solid #EEEDE9; max-height:none; } }
#e1-arch .desc { padding:16px 20px 6px; }
#e1-arch .desc p { font-size:13pt; line-height:1.6; color:#333; margin:0 0 12px 0; }
#e1-arch .desc ul { margin:6px 0 10px 18px; padding-left:0; font-size:13pt; line-height:1.6; color:#333; }
#e1-arch .desc li { margin-bottom:4px; }
#e1-arch .chg { border-left:4px solid #1B5162; background:rgba(27,81,98,.06); border-radius:0 6px 6px 0; padding:10px 14px; margin:0 0 12px 0; font-size:12.5pt; line-height:1.55; }
#e1-arch .chg ul { margin:6px 0 0 20px; }
#e1-arch .code { font-size:13px; line-height:1.65; padding:10px 0; max-height:480px; overflow-y:auto; overflow-x:hidden; background:#fcfcfb; font-family:ui-monospace,Menlo,Consolas,monospace; color:#383a42; }
#e1-arch .ln { display:flex; white-space:pre-wrap; overflow-wrap:anywhere; border-left:3px solid transparent; transition:opacity .2s, background .2s; }
#e1-arch .ln > span:last-child { flex:1; min-width:0; padding-right:16px; }
#e1-arch .ln .g { width:40px; flex:none; text-align:right; padding-right:14px; color:#C4CBCE; user-select:none; }
#e1-arch .ln.add { background:rgba(0,169,114,.13); border-left-color:#00A972; }
#e1-arch .ln.add .g { color:#00A972; font-weight:700; }
#e1-arch .code.focus .ln { opacity:.32; }
#e1-arch .code.focus .ln.hit { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
#e1-arch .code.focus .ln.hit.add { background:rgba(0,169,114,.16); border-left-color:#00A972; }
#e1-arch .t-k { color:#a626a4 } #e1-arch .t-f { color:#4078f2 } #e1-arch .t-s { color:#50a14f } #e1-arch .t-n { color:#986801 } #e1-arch .t-c { color:#a0a1a7; font-style:italic } #e1-arch .t-d { color:#c18401 } #e1-arch .t-y { color:#e45649 }
</style>
<div id="e1-arch">
  <div class="dia"><svg id="e1-svg"></svg></div>
  <div class="panel"><div class="tabs" id="e1-tabs"></div><div class="split" id="e1-split"><div class="side" id="e1-desc"></div><div class="code" id="e1-code"></div></div></div>
</div>
<script>
(function () {
  var RAW = {"C1": [{"name": "agent.py", "desc": "<p>This file defines the agent and an async function to run it. The key integration point is <code>set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))<\/code>, which routes all model calls through Databricks instead of the public OpenAI service.<\/p><p><strong>Key components:<\/strong><\/p><ul><li><code>Agent<\/code> and <code>Runner<\/code>: Define and execute the agent workflow from the OpenAI Agents SDK<\/li><li><code>AsyncDatabricksOpenAI<\/code>: An async client that calls Unity Gateway model services using the OpenAI-compatible interface<\/li><li><code>set_default_openai_api(\"responses\")<\/code>: Sends requests through the Responses API, which supports tool calls with reasoning models<\/li><li><code>MODEL_SERVICE<\/code>: The model service name, read from an environment variable injected by the DAB at deploy time<\/li><\/ul>", "changes": "", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport os\n\n# Read the model service name injected by Databricks Apps\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\n\n# Point the OpenAI Agents SDK at Databricks instead of OpenAI\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Define the agent\nagent = Agent(\n    name=\"Simple Chatbot\",\n    instructions=\"You are a helpful assistant. Answer the user's questions clearly and concisely.\",\n    model=MODEL_SERVICE,\n)\n\n\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent with a list of messages and return the final reply.\"\"\"\n    result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "server.py", "desc": "<p>This FastAPI server exposes two endpoints: <code>GET /<\/code> returns a chat HTML page, and <code>POST /invocations<\/code> forwards conversation messages to the agent and returns the reply. It listens on the port Databricks Apps sets in <code>DATABRICKS_APP_PORT<\/code>.<\/p><p>This demo uses a small hand-written server so each piece is visible. For production agents, the <a href=\"https://github.com/databricks/app-templates/tree/main/agent-openai-agents-sdk\" target=\"_blank\" style=\"color: #1976d2; text-decoration: underline;\">agent app template<\/a> uses MLflow <code>AgentServer<\/code>, which adds the standard Responses API, streaming, and a built-in chat UI.<\/p>", "changes": "", "code": "import os\nimport uvicorn\nfrom pathlib import Path\n\nfrom fastapi import FastAPI\nfrom fastapi.responses import HTMLResponse\nfrom pydantic import BaseModel\n\nfrom agent import run_agent\n\napp = FastAPI()\n\nCHAT_UI = Path(__file__).parent / \"chat.html\"\n\n\nclass ChatRequest(BaseModel):\n    input: list[dict]\n\n\n@app.get(\"/\", response_class=HTMLResponse)\nasync def root():\n    return CHAT_UI.read_text()\n\n\n@app.post(\"/invocations\")\nasync def invocations(request: ChatRequest):\n    reply = await run_agent(request.input)\n    return {\"output\": reply}\n\n\ndef main():\n    uvicorn.run(\"server:app\", host=\"0.0.0.0\", port=int(os.getenv(\"DATABRICKS_APP_PORT\", 8000)))\n\n\nif __name__ == \"__main__\":\n    main()"}, {"name": "pyproject.toml", "desc": "<p>Standard Python project metadata and dependency declaration. The <code>[project.scripts]<\/code> entry creates the <code>start-server<\/code> CLI command referenced by <code>app.yaml<\/code>.<\/p>", "changes": "", "code": "[project]\nname = \"simple-chatbot\"\nversion = \"0.1.0\"\nrequires-python = \">=3.11\"\ndependencies = [\n    \"openai-agents>=0.6.0\",\n    \"mcp<2\",\n    \"databricks-openai==0.17.1\",\n    \"databricks-sdk==0.145.0\",\n    \"fastapi>=0.115.0\",\n    \"uvicorn>=0.34.0\",\n    \"uv\",\n]\n\n[build-system]\nrequires = [\"hatchling\"]\nbuild-backend = \"hatchling.build\"\n\n[tool.hatch.build.targets.wheel]\ninclude = [\"agent.py\", \"server.py\"]\n\n[project.scripts]\nstart-server = \"server:main\""}, {"name": "app.yaml", "desc": "<p>The Databricks App manifest. It declares which environment variables to inject (resolved from DAB resources) and the startup command.<\/p>", "changes": "", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}, {"name": "databricks.yml", "desc": "<p>The Declarative Automation Bundle (DAB) manifest. It names the app, grants it <code>EXECUTE<\/code> on the course model service <code>workspace.agent_deployment.agent-model<\/code>, and targets the <code>dev<\/code> deployment mode. The app name uses bundle substitutions (<code>${workspace.current_user.id}<\/code>, <code>${bundle.target}<\/code>) so each user gets a unique app.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file is only needed for DABs-based deployment. If you are deploying via the Apps UI instead, skip this file and configure the app resources manually on the app Overview page.<\/div>", "changes": "", "code": "bundle:\n  name: simple_chatbot\n\nresources:\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n\ntargets:\n  dev:\n    mode: development\n    default: true"}, {"name": "requirements.txt", "desc": "<p>Databricks Apps automatically runs <code>pip install -r requirements.txt<\/code> during deployment. Installing <code>uv<\/code> here ensures the <code>uv run start-server<\/code> command in <code>app.yaml<\/code> works correctly.<\/p>", "changes": "", "code": "uv"}], "D1": [{"name": "agent.py", "desc": "<p>Replace the contents of <strong>agent.py<\/strong> with the version below. The additions are: the <code>mlflow<\/code> import, tracing setup (<code>set_trace_processors([])<\/code> and <code>mlflow.openai.autolog()<\/code>), the experiment from an environment variable, and the <code>@mlflow.trace<\/code> decorator on <code>run_agent()<\/code>.<\/p>", "changes": "<strong>What changed:<\/strong><ul><li><code>set_trace_processors([])<\/code>: Stops the Agents SDK from sending its own traces to OpenAI<\/li><li><code>mlflow.openai.autolog()<\/code>: Records each Agents SDK step, such as the model call, as a child span<\/li><li><code>mlflow.set_experiment()<\/code>: Targets the experiment created by the DAB, using the ID injected via environment variable<\/li><li><code>@mlflow.trace<\/code>: Wraps every call in a root span that captures inputs, outputs, and timing<\/li><li><code>mlflow.update_current_trace(tags=...)<\/code>: Tags each trace with the agent and model service, so you can filter traces later<\/li><\/ul>", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom agents.tracing import set_trace_processors\nfrom databricks_openai import AsyncDatabricksOpenAI\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Send traces to MLflow instead of OpenAI\nset_trace_processors([])\nmlflow.set_tracking_uri(\"databricks\")\nmlflow.openai.autolog()\n\n# Read environment variables injected by the DAB\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\nmlflow.set_experiment(experiment_id=os.getenv(\"MLFLOW_EXPERIMENT_ID\"))\n\n# Define the agent\nagent = Agent(\n    name=\"Simple Chatbot\",\n    instructions=\"You are a helpful assistant. Answer the user's questions clearly and concisely.\",\n    model=MODEL_SERVICE,\n)\n\n\n@mlflow.trace(name=\"mlflow_chatbot_run\", span_type=\"AGENT\")\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent and return the final reply. The trace records each step.\"\"\"\n    mlflow.update_current_trace(tags={\"agent_name\": agent.name, \"model_service\": MODEL_SERVICE})\n    result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "databricks.yml", "desc": "<p>Replace the contents of <strong>databricks.yml<\/strong> with the version below. This adds an MLflow experiment as a bundle-managed resource and grants the app permission to write to it.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the experiment resource manually on the app Overview page during the deploy step.<\/div>", "changes": "<strong>What changed:<\/strong><ul><li><code>resources.experiments.chatbot_experiment<\/code>: Declares an MLflow experiment that DABs creates on deploy. This is a new resource added to the same app.<\/li><li><code>app-experiment<\/code> resource entry: Grants the app's service principal <code>CAN_EDIT<\/code> on the experiment so it can log traces<\/li><li>The app description updated to reflect the new capability. The bundle name and app name remain the same; redeploying updates the existing app in place.<\/li><\/ul>", "code": "bundle:\n  name: simple_chatbot\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n\ntargets:\n  dev:\n    mode: development\n    default: true"}, {"name": "app.yaml", "desc": "<p>Replace the contents of <strong>app.yaml<\/strong> with the version below. This injects the MLflow experiment ID as an environment variable.<\/p>", "changes": "<strong>What changed:<\/strong><ul><li>The new <code>MLFLOW_EXPERIMENT_ID<\/code> entry uses <code>valueFrom<\/code> to resolve the experiment ID from the <code>app-experiment<\/code> resource declared in <code>databricks.yml<\/code>. The agent reads this at startup to target the correct experiment.<\/li><\/ul>", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}], "E1": [{"name": "agent.py", "desc": "<p>Replace the contents of <strong>agent.py<\/strong> with the final version below. This adds MCP imports, a <code>WorkspaceClient<\/code> for authentication, and the URL of the Databricks-provided SQL MCP <code>system.ai.dbsql<\/code>. <code>run_agent()<\/code> now opens an <code>McpServer<\/code> for each request and passes it to the agent.<\/p>", "changes": "<strong>What changed from Section D:<\/strong><ul><li><code>McpServer<\/code> and <code>WorkspaceClient<\/code> imports added<\/li><li><code>LISTINGS_TABLE<\/code>: Your table's full name, injected by its app resource. The instructions name it, so the agent knows what to query.<\/li><li><code>async with McpServer(...)<\/code>: Connects to <code>system.ai.dbsql<\/code> through Unity Gateway for each request and closes the connection after. Its <code>execute_sql<\/code> tool runs SQL on a SQL warehouse.<\/li><li><code>mcp_servers=[mcp_server]<\/code> on the Agent: The SDK lists the MCP's tools at runtime<\/li><\/ul>", "code": "from agents import Agent, Runner, set_default_openai_api, set_default_openai_client\nfrom agents.tracing import set_trace_processors\nfrom databricks.sdk import WorkspaceClient\nfrom databricks_openai import AsyncDatabricksOpenAI\nfrom databricks_openai.agents import McpServer\nimport mlflow\nimport os\n\n# Point the OpenAI Agents SDK at Databricks\nset_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))\nset_default_openai_api(\"responses\")\n\n# Send traces to MLflow instead of OpenAI\nset_trace_processors([])\nmlflow.set_tracking_uri(\"databricks\")\nmlflow.openai.autolog()\n\n# Read environment variables injected by the DAB\nMODEL_SERVICE = os.getenv(\"MODEL_SERVICE\")\nmlflow.set_experiment(experiment_id=os.getenv(\"MLFLOW_EXPERIMENT_ID\"))\n\n# MCP configuration: the table's full name comes from its app resource\nLISTINGS_TABLE = os.getenv(\"LISTINGS_TABLE\")\n\nws = WorkspaceClient()\n\n# The Databricks-provided SQL MCP, reached through Unity Gateway\nDBSQL_URL = f\"{ws.config.host}/ai-gateway/mcp-services/system.ai.dbsql\"\n\nINSTRUCTIONS = (\n    \"You are a helpful assistant. Answer the user's questions clearly and concisely. \"\n    f\"For questions about San Francisco Airbnb listings, query {LISTINGS_TABLE} with the execute_sql tool.\"\n)\n\n\n@mlflow.trace(name=\"tool_calling_chatbot_run\", span_type=\"AGENT\")\nasync def run_agent(messages: list[dict]) -> str:\n    \"\"\"Run the agent with MCP tools and return the final reply.\"\"\"\n    mlflow.update_current_trace(tags={\"agent_name\": \"Tool-calling Chatbot\", \"model_service\": MODEL_SERVICE})\n    # Connect to the MCP for this request; leaving the block closes the connection\n    async with McpServer(url=DBSQL_URL, workspace_client=ws, name=\"dbsql\", timeout=60.0) as mcp_server:\n        # Define the agent with MCP tool access\n        agent = Agent(\n            name=\"Simple Chatbot\",\n            instructions=INSTRUCTIONS,\n            model=MODEL_SERVICE,\n            mcp_servers=[mcp_server],\n        )\n        result = await Runner.run(agent, messages)\n    return result.final_output"}, {"name": "app.yaml", "desc": "<p>Replace the contents of <strong>app.yaml<\/strong> with the final version below. This adds a <code>LISTINGS_TABLE<\/code> environment variable with the full name of your listings table.<\/p>", "changes": "<strong>What changed from Section D:<\/strong><ul><li>The new entry uses <code>valueFrom<\/code> to read the table's full name from the <code>listings-table<\/code> resource, so there is nothing to type in.<\/li><\/ul>", "code": "env:\n  - name: MODEL_SERVICE\n    valueFrom: model-service\n  - name: MLFLOW_EXPERIMENT_ID\n    valueFrom: app-experiment\n  - name: LISTINGS_TABLE\n    valueFrom: listings-table\n\ncommand: [\"uv\", \"run\", \"start-server\"]"}, {"name": "databricks.yml", "desc": "<p>Replace the contents of <strong>databricks.yml<\/strong> with the final version below. This adds a <code>variables<\/code> block for your catalog and the SQL warehouse, and two resources: <code>SELECT<\/code> on your listings table and <code>CAN_USE<\/code> on the warehouse.<\/p><div class=\"callout\" style=\"border-left:4px solid #1976d2;background:rgba(25,118,210,0.06);padding:10px 12px;margin-top:10px;font-size:14pt;line-height:1.6;border-radius:4px;\"><strong>Not in a Git folder?<\/strong> This file update is only needed for DABs-based deployment. If you are using the Apps UI, skip this and add the table and SQL warehouse resources manually on the app Overview page during the deploy step.<\/div>", "changes": "<strong>What changed from Section D:<\/strong><ul><li><code>variables<\/code> block: <code>catalog_name<\/code> for your catalog, and <code>warehouse_id<\/code>, which looks up the <code>shared_warehouse<\/code> SQL warehouse by name<\/li><li><code>listings-table<\/code> resource: Grants the app’s service principal <code>SELECT<\/code> on <code>sf_airbnb_listings<\/code><\/li><li><code>sql-warehouse<\/code> resource: Grants <code>CAN_USE<\/code> on the warehouse, where the SQL MCP runs the agent’s queries<\/li><li>No grant is needed for <code>system.ai.dbsql<\/code> itself: all workspace users can already call the Databricks-provided MCPs<\/li><\/ul>", "code": "bundle:\n  name: simple_chatbot\n\nvariables:\n  catalog_name:\n    description: \"Per-user lab catalog, e.g. labuser_matthew_mccoy\"\n    default: # <---- Update with your catalog name\n  warehouse_id:\n    description: \"SQL warehouse the SQL MCP runs queries on\"\n    lookup:\n      warehouse: \"shared_warehouse\"\n\nresources:\n  experiments:\n    chatbot_experiment:\n      name: /Users/${workspace.current_user.userName}/simple-chatbot-experiment\n  apps:\n    simple_chatbot:\n      name: agent-${workspace.current_user.id}-${bundle.target}\n      description: \"Simple chatbot agent\"\n      source_code_path: ./\n      resources:\n        - name: model-service\n          uc_securable:\n            securable_full_name: workspace.agent_deployment.agent-model\n            securable_type: MODEL_SERVICE\n            permission: EXECUTE\n        - name: app-experiment\n          experiment:\n            experiment_id: ${resources.experiments.chatbot_experiment.id}\n            permission: CAN_EDIT\n        - name: listings-table\n          uc_securable:\n            securable_full_name: ${var.catalog_name}.agent_deployment.sf_airbnb_listings\n            securable_type: TABLE\n            permission: SELECT\n        - name: sql-warehouse\n          sql_warehouse:\n            id: ${var.warehouse_id}\n            permission: CAN_USE\n\ntargets:\n  dev:\n    mode: development\n    default: true"}]}, STAGE = 2, P = "e1";
  var ORDER = ["agent.py", "server.py", "pyproject.toml", "app.yaml", "databricks.yml", "requirements.txt"];
  var LANG = { py: "python", toml: "toml", yaml: "yaml", yml: "yaml", txt: "text" };
  var PYPROJECT_DESC = '<p>Add <code>"mlflow==3.16.1"</code> to the <code>dependencies</code> list in <strong>pyproject.toml</strong>. Without this dependency, the app cannot import <code>mlflow</code> at runtime and will crash on startup.</p>';
  function buildStages() {
    function toMap(list, key) { var m = {}; list.forEach(function (f) { m[f.name] = f[key]; }); return m; }
    var s1 = toMap(RAW.C1, "code"), s2 = Object.assign({}, s1, toMap(RAW.D1, "code"));
    s2["pyproject.toml"] = s1["pyproject.toml"].replace('    "uv",', '    "uv",\n    "mlflow==3.16.1",');
    var s3 = Object.assign({}, s2, toMap(RAW.E1, "code"));
    var metas = [RAW.C1, RAW.D1, RAW.E1].map(function (list) { var m = {}; list.forEach(function (f) { m[f.name] = { desc: f.desc || "", changes: f.changes || "" }; }); return m; });
    metas[1]["pyproject.toml"] = { desc: PYPROJECT_DESC, changes: "" };
    var stages = [s1, s2, s3].map(function (files, i) { return { files: files, meta: metas[i] }; });
    stages.forEach(function (st, i) {
      var prev = i ? stages[i - 1].files : {};
      st.prev = prev;
      st.changed = ORDER.filter(function (n) { return prev[n] !== st.files[n]; });
    });
    return stages;
  }
  function langOf(name) { return LANG[name.split(".").pop()] || "text"; }
  function esc(s) { return s.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;"); }
  function hlLine(line, lang) {
    var re;
    if (lang === "python") re = /(#.*$)|("""[\s\S]*?"""|"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')|(\b(?:from|import|def|class|return|if|elif|else|for|with|as|async|await|None|True|False|and|or|not|in)\b)|(@[\w.]+)|(\b\w+(?=\())|(\b\d+\.?\d*\b)/g;
    else if (lang === "yaml" || lang === "toml") re = /(#.*$)|("(?:[^"\\]|\\.)*")|(\$\{[^}]+\})|(^\s*-?\s*)([\w][\w\-.]*)(?=\s*[:=])|(\[[\w.\-]+\])/g;
    else return esc(line);
    var out = "", last = 0, m;
    while ((m = re.exec(line))) {
      out += esc(line.slice(last, m.index));
      if (lang === "python") {
        var cls = m[1] ? "c" : m[2] ? "s" : m[3] ? "k" : m[4] ? "d" : m[5] ? "f" : "n";
        out += '<span class="t-' + cls + '">' + esc(m[0]) + "</span>";
      } else if (m[1]) out += '<span class="t-c">' + esc(m[0]) + "</span>";
      else if (m[2]) out += '<span class="t-s">' + esc(m[0]) + "</span>";
      else if (m[3]) out += '<span class="t-d">' + esc(m[0]) + "</span>";
      else if (m[5]) out += esc(m[4]) + '<span class="t-y">' + esc(m[5]) + "</span>";
      else out += '<span class="t-k">' + esc(m[0]) + "</span>";
      last = m.index + m[0].length;
      if (m[0] === "") re.lastIndex++;
    }
    return out + esc(line.slice(last));
  }
  function addedSet(a, b) {
    a = a ? a.split("\n") : []; b = b.split("\n");
    var n = a.length, m = b.length, dp = [], i, j, set = {}, k = 0;
    for (i = 0; i <= n; i++) dp.push(new Array(m + 1).fill(0));
    for (i = n - 1; i >= 0; i--) for (j = m - 1; j >= 0; j--) dp[i][j] = a[i] === b[j] ? dp[i + 1][j + 1] + 1 : Math.max(dp[i + 1][j], dp[i][j + 1]);
    i = 0; j = 0;
    while (j < m) {
      if (i < n && a[i] === b[j]) { i++; j++; }
      else if (i < n && dp[i + 1][j] >= dp[i][j + 1]) i++;
      else set[j++] = 1;
    }
    return set;
  }
  var COPY_ICON = '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>';
  function copyText(text, btn) {
    var t = document.createElement("textarea"); t.value = text; t.style.position = "fixed"; t.style.opacity = "0"; document.body.appendChild(t); t.select();
    var ok = false; try { ok = document.execCommand("copy"); } catch (e) {} document.body.removeChild(t);
    if (!ok && navigator.clipboard) { navigator.clipboard.writeText(text); ok = true; }
    if (ok) { btn.classList.add("ok"); btn.innerHTML = "Copied"; setTimeout(function () { btn.classList.remove("ok"); btn.innerHTML = COPY_ICON + "Copy"; }, 1500); }
  }
  var S = buildStages(), st = S[STAGE];
  var CONTS = [
    { id: "app", x: 215, y: 84, w: 505, h: 296, label: "DATABRICKS APP" },
    { id: "gw", x: 790, y: 40, w: 270, h: STAGE < 2 ? 180 : 340, label: "UNITY GATEWAY" }
  ];
  var NODES = [
    { id: "browser", x: 14, y: 170, w: 122, h: 84, t: "Browser", s: "chat.html", from: 0 },
    { id: "server", x: 240, y: 170, w: 175, h: 84, t: "server.py", s: "FastAPI", from: 0, mono: 1 },
    { id: "agent", x: 522, y: 170, w: 175, h: 84, t: "agent.py", s: "OpenAI Agents SDK", from: 0, mono: 1 },
    { id: "runtime", x: 240, y: 292, w: 175, h: 62, t: "uv run start-server", s: "", from: 0, mono: 1, small: 1 },
    { id: "model", x: 815, y: 104, w: 220, h: 84, t: "agent-model", s: "Model service", from: 0, perm: "EXECUTE", mono: 1 },
    { id: "mcp", x: 815, y: 264, w: 220, h: 84, t: "system.ai.dbsql", s: "MCP server", from: 2, mono: 1 },
    { id: "warehouse", x: 1100, y: 264, w: 200, h: 84, t: "shared_warehouse", s: "SQL warehouse", from: 2, perm: "CAN_USE", mono: 1 },
    { id: "table", x: 1100, y: 416, w: 200, h: 84, t: "sf_airbnb_listings", s: "Unity Catalog table", from: 2, perm: "SELECT", mono: 1 },
    { id: "exp", x: 479, y: 420, w: 262, h: 84, t: "MLflow experiment", s: ["", "Traces", "Traces + tool spans"][STAGE], from: 1, perm: "CAN_EDIT" }
  ];
  var EDGES = [
    { a: "browser", b: "server", l: "/invocations", from: 0 },
    { a: "server", b: "agent", l: "run_agent()", from: 0 },
    { a: "agent", b: "model", l: "Responses API", from: 0, end: 1 },
    { a: "agent", b: "mcp", l: "execute_sql", from: 2, end: 1 },
    { a: "mcp", b: "warehouse", l: "SQL", from: 2 },
    { a: "warehouse", b: "table", l: "", from: 2, down: 1 },
    { a: "agent", b: "exp", l: "", from: 1, down: 1 }
  ];
  // [file, line prefix, include the indented block that follows]
  var MAP = {
    browser: [["server.py", "CHAT_UI"], ["server.py", "@app.get"], ["server.py", "async def root"], ["server.py", "return CHAT_UI"]],
    server: [["server.py", "import uvicorn"], ["server.py", "from fastapi"], ["server.py", "from pydantic"], ["server.py", "from agent import"], ["server.py", "app = FastAPI"],
      ["server.py", "class ChatRequest", 1], ["server.py", "@app.post"], ["server.py", "async def invocations", 1], ["server.py", "def main", 1]],
    agent: [["agent.py", "from agents import"], ["agent.py", "# Define"], ["agent.py", "agent = Agent(", 1], ["agent.py", "async def run_agent"], ["agent.py", "result = await Runner.run"], ["agent.py", "return"], ["agent.py", "final_output = result"]],
    runtime: [["app.yaml", "command:"], ["pyproject.toml", "dependencies = [", 1], ["pyproject.toml", "[tool.hatch"], ["pyproject.toml", "include ="], ["pyproject.toml", "[project.scripts]"], ["pyproject.toml", "start-server"], ["requirements.txt", "uv"]],
    app: [["databricks.yml", "bundle:", 1], ["databricks.yml", "apps:"], ["databricks.yml", "simple_chatbot:"], ["databricks.yml", "name: agent-"], ["databricks.yml", "description:"], ["databricks.yml", "source_code_path:"], ["databricks.yml", "resources:"], ["databricks.yml", "targets:", 1]],
    gw: [["agent.py", "set_default_openai_client"], ["agent.py", "DBSQL_URL ="], ["agent.py", "# The Databricks-provided"]],
    model: [["agent.py", "from databricks_openai import AsyncDatabricksOpenAI"], ["agent.py", "# Point the"], ["agent.py", "set_default_openai_client"], ["agent.py", "set_default_openai_api"], ["agent.py", "# Read the model"],
      ["agent.py", "MODEL_SERVICE ="], ["agent.py", "model=MODEL_SERVICE"], ["app.yaml", "- name: MODEL_SERVICE", 1], ["databricks.yml", "- name: model-service", 1]],
    exp: [["agent.py", "from agents.tracing"], ["agent.py", "import mlflow"], ["agent.py", "# Send traces"], ["agent.py", "set_trace_processors("], ["agent.py", "mlflow.set_tracking_uri"], ["agent.py", "mlflow.openai.autolog"], ["agent.py", "mlflow.set_experiment"],
      ["agent.py", "@mlflow.trace"], ["agent.py", "mlflow.update_current_trace"], ["app.yaml", "- name: MLFLOW_EXPERIMENT_ID", 1], ["databricks.yml", "experiments:", 1], ["databricks.yml", "- name: app-experiment", 1], ["pyproject.toml", "\"mlflow=="]],
    mcp: [["agent.py", "from databricks_openai.agents"], ["agent.py", "from databricks.sdk"], ["agent.py", "ws = WorkspaceClient"], ["agent.py", "# The Databricks-provided"], ["agent.py", "DBSQL_URL ="], ["agent.py", "# Connect to the MCP"], ["agent.py", "async with McpServer("], ["agent.py", "mcp_servers="]],
    warehouse: [["databricks.yml", "warehouse_id:", 1], ["databricks.yml", "- name: sql-warehouse", 1]],
    table: [["agent.py", "# MCP configuration"], ["agent.py", "LISTINGS_TABLE ="], ["agent.py", "f\"For questions"], ["app.yaml", "- name: LISTINGS_TABLE", 1], ["databricks.yml", "catalog_name:", 1], ["databricks.yml", "- name: listings-table", 1]]
  };
  var sel = null, file = st.changed[0];
  function $(id) { return document.getElementById(P + "-" + id); }
  function ind(l) { return l.length - l.replace(/^\s+/, "").length; }
  function hits(id, name) {
    var lines = st.files[name].split("\n"), set = {};
    (MAP[id] || []).forEach(function (m) {
      if (m[0] !== name) return;
      lines.forEach(function (l, i) {
        if (l.trim().indexOf(m[1]) !== 0) return;
        set[i] = 1;
        if (!m[2]) return;
        var base = ind(l), j = i + 1;
        while (j < lines.length && (!lines[j].trim() || ind(lines[j]) > base)) { if (lines[j].trim()) set[j] = 1; j++; }
        if (j < lines.length && /^[\)\]]/.test(lines[j].trim()) && ind(lines[j]) === base) set[j] = 1;
      });
    });
    return set;
  }
  function count(o) { return Object.keys(o).length; }
  function N(id) { return NODES.filter(function (n) { return n.id === id; })[0]; }
  function vis(o) { return o.from <= STAGE; }
  function isNew(o) { return STAGE > 0 && o.from === STAGE; }
  function draw() {
    var h = '<defs><marker id="' + P + '-ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#9AABB1"/></marker>' +
      '<marker id="' + P + '-arn" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#FF5F46"/></marker></defs>';
    CONTS.forEach(function (c) {
      h += '<g class="cont' + (sel === c.id ? " sel" : "") + '"><rect class="box" x="' + c.x + '" y="' + c.y + '" width="' + c.w + '" height="' + c.h + '" rx="16"/>' +
        '<g class="hdr" data-id="' + c.id + '"><rect x="' + c.x + '" y="' + c.y + '" width="' + c.w + '" height="40" rx="16" fill="transparent"/><text x="' + (c.x + 22) + '" y="' + (c.y + 27) + '">' + c.label + "</text></g></g>";
    });
    EDGES.forEach(function (e) {
      if (!vis(e)) return;
      var a = N(e.a), b = N(e.b), d, lx, x1, y1, x2, y2;
      if (e.down) { x1 = a.x + a.w / 2; y1 = a.y + a.h; d = "M" + x1 + "," + y1 + " L" + x1 + "," + (b.y - 2); }
      else {
        x1 = a.x + a.w; y1 = a.y + a.h / 2; x2 = b.x; y2 = b.y + b.h / 2; lx = (x1 + x2) / 2;
        d = "M" + x1 + "," + y1 + " C" + lx + "," + y1 + " " + lx + "," + y2 + " " + (x2 - 2) + "," + y2;
      }
      var lab = !e.l ? "" : e.end ? '<text x="' + (b.x + 4) + '" y="' + (b.y - 9) + '">' + e.l + "</text>" : '<text x="' + lx + '" y="' + (y1 - 10) + '" text-anchor="middle">' + e.l + "</text>";
      h += '<g class="ed' + (isNew(e) ? " new" : "") + '"><path d="' + d + '" marker-end="url(#' + P + (isNew(e) ? "-arn" : "-ar") + ')"/>' + lab + "</g>";
    });
    NODES.forEach(function (n) {
      if (!vis(n)) return;
      var cy = n.y + n.h / 2;
      h += '<g class="nd' + (n.mono ? " mono" : "") + (n.small ? " small" : "") + (isNew(n) ? " new" : "") + (sel === n.id ? " sel" : "") + '" data-id="' + n.id + '">' +
        '<rect class="box" x="' + n.x + '" y="' + n.y + '" width="' + n.w + '" height="' + n.h + '" rx="10"/>' +
        '<text class="t" x="' + (n.x + n.w / 2) + '" y="' + (n.s ? cy - 4 : cy + 5) + '" text-anchor="middle">' + n.t + "</text>" +
        (n.s ? '<text class="s" x="' + (n.x + n.w / 2) + '" y="' + (cy + 17) + '" text-anchor="middle">' + n.s + "</text>" : "");
      if (n.perm) { var pw = n.perm.length * 7.6 + 16; h += '<g class="pill"><rect x="' + (n.x + n.w - pw - 10) + '" y="' + (n.y - 10) + '" width="' + pw + '" height="20" rx="10"/><text x="' + (n.x + n.w - 10 - pw / 2) + '" y="' + (n.y + 4) + '" text-anchor="middle">' + n.perm + "</text></g>"; }
      h += "</g>";
    });
    var svg = $("svg");
    svg.setAttribute("viewBox", "0 30 " + (STAGE < 2 ? 1080 : 1320) + " " + (STAGE < 1 ? 360 : 486));
    svg.innerHTML = h;
    Array.prototype.forEach.call(svg.querySelectorAll("[data-id]"), function (g) { g.addEventListener("click", function () { pick(g.getAttribute("data-id")); }); });
  }
  function pick(id) {
    sel = sel === id ? null : id;
    if (sel && !count(hits(sel, file))) {
      var withHits = ORDER.filter(function (n) { return count(hits(sel, n)); });
      file = withHits.filter(function (n) { return st.changed.indexOf(n) >= 0; })[0] || withHits[0] || file;
    }
    render();
  }
  function render() {
    draw();
    var t = $("tabs");
    t.innerHTML = ORDER.map(function (n) {
      var ch = st.changed.indexOf(n) >= 0, k = sel ? count(hits(sel, n)) : 0;
      return '<div class="tab' + (n === file ? " on" : "") + (ch ? "" : " same") + '" data-f="' + n + '">' + n + (STAGE && ch ? '<span class="d"></span>' : "") + (k ? '<span class="n">' + k + "</span>" : "") + "</div>";
    }).join("") + '<span class="grow"></span><button class="cp" id="' + P + '-cp">' + COPY_ICON + "Copy</button>";
    Array.prototype.forEach.call(t.querySelectorAll(".tab"), function (el) { el.addEventListener("click", function () { file = el.getAttribute("data-f"); render(); }); });
    var meta = st.changed.indexOf(file) >= 0 && st.meta[file];
    $("desc").innerHTML = meta ? '<div class="desc">' + meta.desc + (meta.changes ? '<div class="chg">' + meta.changes + "</div>" : "") + "</div>" : "";
    $("split").className = "split" + (meta ? "" : " full");
    var hs = sel ? hits(sel, file) : null, add = STAGE && st.changed.indexOf(file) >= 0 ? addedSet(st.prev[file], st.files[file]) : {};
    var c = $("code");
    c.className = "code" + (hs && count(hs) ? " focus" : "");
    c.innerHTML = st.files[file].split("\n").map(function (l, i) {
      return '<div class="ln' + (add[i] ? " add" : "") + (hs && hs[i] ? " hit" : "") + '"><span class="g">' + (add[i] ? "+" : i + 1) + "</span><span>" + (hlLine(l, langOf(file)) || " ") + "</span></div>";
    }).join("");
    c.style.maxHeight = ""; /* height now follows the description panel via CSS grid stretch */
    var first = c.querySelector(".hit"); if (first) c.scrollTop = first.offsetTop - c.offsetTop - 40;
    var b = $("cp"); b.onclick = function () { copyText(st.files[file], b); };
  }
  render();
})();
</script>

### E2. Redeploy

Redeploy using whichever method you used in Sections C2 and D2. This update adds two resources, your listings table and a SQL warehouse, so the agent can query the table through the SQL MCP. Select a card below.

<br></br>

<div style="width: 100%; font-family: sans-serif;">

<style>
.e2d-row { padding: 4px 6px; box-sizing: border-box; display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
.e2d-box {
  display: block; background: #fff; border: 2px solid #C6DCEA; box-sizing: border-box; border-radius: 12px;
  padding: 16px 20px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);
  cursor: pointer; user-select: none; text-align: left;
  transition: transform 0.12s, box-shadow 0.15s, border-color 0.15s;
}
.e2d-box::before { content: ""; display: block; height: 4px; border-radius: 2px; background: var(--pc); margin: -2px 0 14px; }
.e2d-box:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(27,49,57,.10); }
.e2d-box.active { border-color: var(--pc); box-shadow: 0 6px 16px rgba(27,49,57,.10); }
.e2d-pill {
  display: inline-block; background: var(--pc); color: #fff; font-size: 10.5pt; font-weight: 700;
  letter-spacing: 0.06em; border-radius: 999px; padding: 3px 12px; pointer-events: none;
}
.e2d-title { display: block; font-size: 16pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 10px; pointer-events: none; }
.e2d-subtitle { display: block; font-size: 12.5pt; font-weight: 500; color: #5A6F77; line-height: 1.3; margin-top: 3px; pointer-events: none; }
.e2d-detail-wrap {
  padding: 0 6px 8px; box-sizing: border-box; overflow: hidden; max-height: 0; opacity: 0; margin-top: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
}
.e2d-detail-wrap.open { max-height: 2400px; opacity: 1; margin-top: 16px; }
.e2d-detail-card {
  background: #fff; border: 1.5px solid #C6DCEA; border-radius: 12px;
  padding: 18px 22px 20px; box-sizing: border-box; box-shadow: 0 2px 8px rgba(27,49,57,.07);
}
.e2d-accent { height: 4px; border-radius: 2px; margin: -2px 0 14px; }
.e2d-steps-block { font-size: 14.5pt; line-height: 1.6; color: #0b2026; margin-bottom: 14px; }
.e2d-steps-block ol { margin: 8px 0 0 0; padding-left: 22px; }
.e2d-steps-block li { margin-bottom: 8px; }
.e2d-steps-block ul { margin: 4px 0 8px 20px; list-style: disc; }
.e2d-info-block {
  background: #F9F7F4; border-left: 4px solid; border-radius: 0 6px 6px 0;
  padding: 12px 14px; font-size: 14pt; line-height: 1.5; color: #0b2026;
}
</style>

<div>
  <div class="e2d-row">
    <span class="e2d-box active" data-id="0" onclick="e2dSelect(0)" style="--pc:#5A6F77;">
      <span class="e2d-pill">DABS UI</span>
      <span class="e2d-title">Git folder</span>
      <span class="e2d-subtitle">Deploys from databricks.yml</span>
    </span>
    <span class="e2d-box" data-id="1" onclick="e2dSelect(1)" style="--pc:#FF5F46;">
      <span class="e2d-pill">APPS UI</span>
      <span class="e2d-title">No Git folder</span>
      <span class="e2d-subtitle">Add resources by hand</span>
    </span>
  </div>

  <div class="e2d-detail-wrap" id="e2d-detail-wrap">
    <div class="e2d-detail-card" id="e2d-detail-card"></div>
  </div>

</div>

</div>

<script>
var E2D_CARDS = [
  {
    title: "DABs UI (Git folder)",
    color: "#5A6F77",
    bg: "rgba(90,111,119,0.10)",
    infoBg: "rgba(90,111,119,0.10)",
    content: `
      <ol>
        <li>Click the rocket icon on any file in the project folder.</li>
        <li>Click <strong>Deploy</strong>, wait for validation, then click <strong>Deploy</strong> again.</li>
        <li>Navigate to the app and click <strong>Deploy</strong> to push the updated source.</li>
        <li>Notice the new <strong>table</strong> and <strong>SQL warehouse</strong> resources now appear under <strong>App resources</strong>.</li>
        <li>After deployment, click on the URL to open the app.</li>
      </ol>
    `,
    info: '<strong>Requires a Git folder.</strong> The DABs UI detects the new resources in <code>databricks.yml</code> and grants <code>SELECT</code> on the table and <code>CAN_USE</code> on the warehouse automatically. Wait for the app status to return to <strong>Running</strong>.'
  },
  {
    title: "Apps UI (no Git folder)",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    infoBg: "rgba(255,95,70,0.10)",
    content: `
      <ol>
        <li>Navigate to the app Overview page and click <strong>(⋮)</strong> icon then on  <strong>Edit</strong> at the top right.</li>
        <li>Click <strong>resources</strong> at the top of the setup wizard.</li>
        <li>Click <strong>+ Add resource</strong> and select <strong>UC table</strong>.</li>
        <li>Enter the full table name: <code>{catalog}.agent_deployment.sf_airbnb_listings</code> (replace <code>{catalog}</code> with your catalog name).</li>
        <li>Make sure <strong>Permission</strong> is set to <strong>Can select</strong> and <strong>Resource key</strong> is set to <strong>listings-table</strong>.</li>
        <li>Click <strong>+ Add resource</strong> again and select <strong>SQL warehouse</strong>.</li>
        <li>Select <strong>shared_warehouse</strong>. Make sure <strong>Permission</strong> is set to <strong>Can use</strong> and <strong>Resource key</strong> is set to <strong>sql-warehouse</strong>.</li>
        <li>Click <strong>Save</strong>.</li>
        <li>Back on the Overview page, click <strong>Deploy</strong> to push the updated source.</li>
      </ol>
    `,
    info: '<strong>No Git folder required.</strong> You add the table and SQL warehouse resources manually on the Overview page. Wait for the app status to return to <strong>Running</strong>.'
  }
];

var e2dCurrent = null;

function e2dSelect(id) {
  var wrap = document.getElementById('e2d-detail-wrap');
  var card = document.getElementById('e2d-detail-card');
  var c = E2D_CARDS[id];

  document.querySelectorAll('.e2d-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (e2dCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.e2d-box').forEach(function(b) {
      b.classList.remove('active');
    });
    e2dCurrent = null;
    return;
  }

  e2dCurrent = id;

  var label = c.title.split(' (')[0].toUpperCase();
  var sub = c.title.replace(/^.*\((.*)\)$/, '$1'); sub = sub.charAt(0).toUpperCase() + sub.slice(1);
  card.innerHTML =
    '<div class="e2d-accent" style="background:' + c.color + ';"></div>'
    + '<span class="e2d-pill" style="--pc:' + c.color + ';">' + label + '</span>'
    + '<div style="font-size:18pt;font-weight:700;margin:10px 0 12px;color:#0b2026;">' + sub + '</div>'
    + '<div class="e2d-steps-block">'
    + c.content
    + '</div>'
    + '<div class="e2d-info-block" style="border-color:' + c.color + ';">'
    + c.info
    + '</div>';

  wrap.classList.add('open');
}

e2dSelect(0);
</script>

### E3. Test and Verify Tool Call Traces

The agent can now query your listings table through the `system.ai.dbsql` MCP. Verify that tool calls appear in the traces.

1. Open the app URL and send a query that triggers the tool, such as: *"What is the average price for listings in the Mission neighborhood?"*
2. Wait for the agent to respond. The response should include the average price from your table.
3. Click **Experiments** in the left workspace sidebar, then search for and open the experiment whose name is printed by the code cell before D2: `dev your_username simple-chatbot-experiment`.
4. Click the **Traces** tab and open the most recent trace. You should now see:
   - The **root span** (`tool_calling_chatbot_run`) capturing the full agent invocation
   - **`execute_sql` spans**, one for each MCP tool call. Click one to see the query the agent wrote and the result it returned. The agent may first run a quick query to look at the table, then a second to compute the average.
   - **`Response` spans** for the model calls before and after each tool call

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Compare this trace to the one from Section D3. Adding MCP tools added new child spans without any change to the tracing code. <code>mlflow.openai.autolog()</code> records each tool call as part of the trace tree.</p>
</div>
</div>
</div>

## F. Filter Traces by Tag

Send a few more messages through the chat interface, mixing questions that use the tool with ones that don't. Then filter the traces by the tags the app adds.

1. In the experiment, click the **Traces** tab. You should see a trace entry for each message you sent.
2. Filter the list by the tag `agent_name` or `model_service`. Tags like these separate one app's traces from others in the same experiment.
3. Compare a trace that called `execute_sql` with one that didn't. Both share the same root span and tags; only the tool call spans differ.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Traces, not runs</strong>
<p style="margin: 8px 0 0 0; color: #333;">The app records each request as a trace, not as an MLflow run. A trace captures the execution graph of one request: spans, timing, inputs, and outputs. Tags on the trace, such as <code>model_service</code>, let you filter and compare many requests over time.</p>
</div>
</div>
</div>

## G. Clean Up

When you are finished with this demo, remove the resources created during deployment.

1. Navigate to the **Apps** page in the workspace sidebar and select the three vertical dots next to your app. Select **Delete** to stop and remove the app.
2. Navigate to **Experiments**, open the experiment, and delete it if you no longer need its traces.

If you deployed with the Databricks CLI, `databricks bundle destroy` removes the app and the experiment together.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

